In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 59
==================================================

Week: 9 of 24
Day: 59 of 168
Date: December 26, 2024
Topic: User Accounts, History & API Management
Overall Progress: (59/168 days)

Week 9 Progress:
✅ Day 57: Streamlit platform integration (COMPLETED)
✅ Day 58: Job Matcher + advanced features (COMPLETED)
🔄 Day 59: User accounts, history, API keys (TODAY!)
⬜ Day 60: Performance optimization (<2s inference)
⬜ Day 61: Analytics dashboard
⬜ Day 62: Deployment preparation (Docker, config)
⬜ Day 63: Final testing + production repo

Progress: 43% (3/7 days)

==================================================
🎯 Week 9 Project: TextAI Studio Web Platform
==================================================

- Production-grade web platform with user management
- Usage history and analytics
- API key system for developers
- Rate limiting for fair usage
- Professional platform features

🎯 Today's Learning Objectives:

1. Implement user account system (optional authentication)
2. Build usage history tracking (store past analyses)
3. Create API key management for developers
4. Add rate limiting to prevent abuse
5. Design session state management
6. Build user dashboard with analytics

📚 Today's Structure:

Part 1 (1.5h): User Accounts & Authentication
Part 2 (1h): Usage History & Session Management
Part 3 (1h): API Key Management & Rate Limiting
Part 4 (0.5h): Testing, Documentation & Summary

🎯 SUCCESS CRITERIA:

✅ Optional user authentication working
✅ Usage history stored and retrievable
✅ API keys generated and validated
✅ Rate limiting prevents abuse
✅ User dashboard shows analytics
✅ Session state properly managed
✅ All features tested and documented
✅ Ready for Day 60 (performance optimization)

==================================================
"""

In [2]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys
import subprocess

# Install packages properly
packages = ['pyjwt', 'bcrypt', 'streamlit-authenticator']

print("📦 Installing authentication libraries...")
for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"✅ {package} installed")
    except:
        print(f"⚠️ {package} installation failed (may already be installed)")

print("\n✅ Installation complete!")
print("\n" + "="*80)

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "="*80)
print("📚 IMPORTING LIBRARIES")
print("="*80)

import os
import time
from datetime import datetime, timedelta
import json
import hashlib
import secrets
import uuid

# Authentication
try:
    import jwt
    print("✅ jwt imported")
except ImportError:
    print("❌ jwt import failed - installing PyJWT...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyjwt'])
    import jwt
    print("✅ jwt imported after install")

try:
    import bcrypt
    print("✅ bcrypt imported")
except ImportError:
    print("❌ bcrypt import failed - installing...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'bcrypt'])
    import bcrypt
    print("✅ bcrypt imported after install")

# Data handling
import pandas as pd
import pickle

# Streamlit
import streamlit as st

# File operations
from pathlib import Path

print("\n✅ All libraries imported successfully!")
print("="*80)

# ==================================================
# ENVIRONMENT SETUP
# ==================================================

print("\n" + "="*80)
print("🔧 ENVIRONMENT SETUP")
print("="*80)

# Project paths
WEEK_9_DIR = Path(r"C:\Users\audrey\Documents\ml-learning-lab\week_9_streamlit_nlp_platform")

print(f"Week 9 Directory: {WEEK_9_DIR}")

# Create data directories for user management
USER_DATA_DIR = WEEK_9_DIR / "user_data"
USER_DATA_DIR.mkdir(exist_ok=True)

HISTORY_DIR = USER_DATA_DIR / "history"
HISTORY_DIR.mkdir(exist_ok=True)

API_KEYS_DIR = USER_DATA_DIR / "api_keys"
API_KEYS_DIR.mkdir(exist_ok=True)

print(f"\n📁 Data Directories:")
print(f"   User Data: {USER_DATA_DIR}")
print(f"   History: {HISTORY_DIR}")
print(f"   API Keys: {API_KEYS_DIR}")

print("\n✅ Environment setup complete!")
print("="*80)

📦 Installing authentication libraries...
✅ pyjwt installed
✅ bcrypt installed
✅ streamlit-authenticator installed

✅ Installation complete!


📚 IMPORTING LIBRARIES
✅ jwt imported
✅ bcrypt imported

✅ All libraries imported successfully!

🔧 ENVIRONMENT SETUP
Week 9 Directory: C:\Users\audrey\Documents\ml-learning-lab\week_9_streamlit_nlp_platform

📁 Data Directories:
   User Data: C:\Users\audrey\Documents\ml-learning-lab\week_9_streamlit_nlp_platform\user_data
   History: C:\Users\audrey\Documents\ml-learning-lab\week_9_streamlit_nlp_platform\user_data\history
   API Keys: C:\Users\audrey\Documents\ml-learning-lab\week_9_streamlit_nlp_platform\user_data\api_keys

✅ Environment setup complete!


In [3]:
print("\n" + "="*80)
print("👤 PART 1: USER ACCOUNTS & AUTHENTICATION")
print("="*80)


👤 PART 1: USER ACCOUNTS & AUTHENTICATION


In [4]:
# ==================================================
# EXERCISE 1.1: DESIGN USER AUTHENTICATION SYSTEM
# ==================================================

print("\n" + "="*80)
print("EXERCISE 1.1: Planning User Authentication Architecture")
print("="*80)

"""
📖 THEORY: Authentication in Web Applications

What is Authentication?
==================================================

Authentication = Verifying user identity
- Who are you?
- Prove it with credentials
- Get access if valid

Authorization = What can you do?
- What permissions do you have?
- Which features can you access?
- What data can you see?

Authentication Methods:
==================================================

1. Basic Authentication:
   - Username + Password
   - Stored locally
   - Simple but limited

2. Session-Based:
   - Login once
   - Get session token
   - Token expires

3. Token-Based (JWT):
   - JSON Web Tokens
   - Stateless
   - Scalable

4. OAuth/Social Login:
   - Google, GitHub, etc.
   - Third-party auth
   - No password storage

5. API Keys:
   - For programmatic access
   - Long-lived tokens
   - Rate-limitable

Our Approach: Optional Simple Auth
==================================================

Why Optional:
- Demo/portfolio: No login required
- Power users: Can create account
- Developers: Can get API keys
- Best of both worlds

Features:
1. Guest Mode (Default):
   - No login required
   - Full tool access
   - No history saved
   - Session-based only

2. Registered User Mode:
   - Simple signup (username + password)
   - Usage history saved
   - API key generation
   - Analytics dashboard

Password Security:
==================================================

NEVER store plain passwords!

Bad (Plain text):
passwords = {'alice': 'password123'}  # NEVER DO THIS!

Good (Hashed):
```python
import bcrypt

# Hash password
password = 'password123'
hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())

# Verify later
if bcrypt.checkpw(password.encode(), hashed):
    print("Valid!")
```

How bcrypt works:
- One-way hash (can't reverse)
- Salt added (unique per password)
- Slow by design (prevents brute force)
- Industry standard

Session Management in Streamlit:
==================================================

Streamlit Session State:
```python
# Initialize
if 'user' not in st.session_state:
    st.session_state.user = None

# Login
st.session_state.user = {
    'username': 'alice',
    'logged_in': True
}

# Check status
if st.session_state.user and st.session_state.user['logged_in']:
    st.write("Welcome back!")
```

Persistence:
- Session state = During browser session
- For persistence: Save to disk/database
- Our approach: Local JSON files

User Data Structure:
==================================================

User Object:
```python
user = {
    'username': 'alice',
    'password_hash': b'$2b$12$...',  # bcrypt hash
    'email': 'alice@example.com',
    'created_at': '2024-12-26T10:00:00',
    'api_key': 'sk_abc123...',
    'usage_count': 42,
    'last_login': '2024-12-26T15:30:00'
}
```

Storage:
- File: user_data/users.json
- Format: JSON (dict of users)
- Security: Hash passwords only

Architecture:
==================================================

Components:
1. UserManager class
   - register_user()
   - login_user()
   - verify_password()
   - get_user()

2. AuthUI class
   - login_form()
   - signup_form()
   - logout_button()

3. Session Manager
   - init_session()
   - is_logged_in()
   - get_current_user()

Flow:
1. App starts → Check session
2. No user → Show guest mode OR login option
3. User logs in → Verify credentials
4. Valid → Set session state
5. Use app → Track usage
6. Logout → Clear session

Why Simple Auth is Enough:
==================================================

For Portfolio/Demo:
- Shows understanding
- Production-ready concept
- Easy to upgrade later

Not Needed:
- Real database (overkill)
- OAuth (too complex)
- 2FA (unnecessary)
- Email verification (not needed)

Production Upgrade Path:
- Replace JSON → PostgreSQL
- Add OAuth → streamlit-authenticator
- Deploy → Use Auth0, Supabase, etc.
"""

print("\n⏱️  Designing authentication system...")

print("\n🏗️  Authentication Architecture:")

print("\n   Guest Mode (Default):")
print("      • No login required")
print("      • Full tool access")
print("      • No data persistence")
print("      • Session ends when browser closes")
print("      • Perfect for demos and testing")

print("\n   Registered User Mode:")
print("      • Simple signup (username + password)")
print("      • Secure password hashing (bcrypt)")
print("      • Usage history saved to disk")
print("      • API key generation")
print("      • Personal analytics dashboard")

print("\n   Authentication Flow:")
print("      1. App loads")
print("      2. Check st.session_state.user")
print("      3. If None:")
print("         • Show 'Continue as Guest' button")
print("         • Show 'Login' form")
print("         • Show 'Sign Up' form")
print("      4. If logged in:")
print("         • Show username + logout button")
print("         • Enable history tracking")
print("         • Show user dashboard")

print("\n🔐 Security Measures:")

print("\n   Password Security:")
print("      • bcrypt hashing (industry standard)")
print("      • Automatic salt generation")
print("      • Slow by design (prevents brute force)")
print("      • One-way hash (cannot reverse)")
print("      • Cost factor: 12 (2^12 iterations)")

print("\n   Data Storage:")
print("      • Users: user_data/users.json")
print("      • Structure: {username: user_object}")
print("      • Password: Hashed only, never plain")
print("      • API keys: Randomly generated (secrets)")

print("\n   Session Security:")
print("      • Session state cleared on logout")
print("      • No sensitive data in session")
print("      • Username only (no password)")

print("\n📊 User Data Structure:")

user_structure = {
    'username': 'alice',
    'password_hash': 'bcrypt_hash_bytes',
    'email': 'alice@example.com',
    'created_at': '2024-12-26T10:00:00',
    'api_key': 'sk_abc123def456...',
    'usage_count': 42,
    'last_login': '2024-12-26T15:30:00',
    'tools_used': {
        'sentiment': 15,
        'summarizer': 10,
        'fake_news': 12,
        'job_matcher': 5
    }
}

print("\n   User Object:")
for key, value in user_structure.items():
    print(f"      • {key}: {value}")

print("\n💡 Design Decisions:")

print("\n   Why Optional Auth:")
print("      • Portfolio demo needs no barriers")
print("      • Guest mode = instant access")
print("      • Power users can register")
print("      • Developers get API keys")

print("\n   Why Simple (not OAuth):")
print("      • Faster to implement")
print("      • No external dependencies")
print("      • Good enough for demo")
print("      • Easy to upgrade later")

print("\n   Why Local Storage (not DB):")
print("      • Simpler for portfolio project")
print("      • No database setup required")
print("      • JSON files easy to inspect")
print("      • Production: Upgrade to PostgreSQL")

print("\n🎯 Implementation Components:")

print("\n   1. UserManager Class:")
print("      • register_user(username, password, email)")
print("      • login_user(username, password)")
print("      • verify_password(password, hash)")
print("      • get_user(username)")
print("      • save_users()")
print("      • load_users()")

print("\n   2. Authentication UI:")
print("      • Guest mode button")
print("      • Login form (username, password)")
print("      • Signup form (username, email, password)")
print("      • Logout button")
print("      • User info display")

print("\n   3. Session Management:")
print("      • init_session()")
print("      • is_logged_in()")
print("      • get_current_user()")
print("      • logout()")

print("\n✅ Exercise 1.1 Complete!")
print("="*80)


EXERCISE 1.1: Planning User Authentication Architecture

⏱️  Designing authentication system...

🏗️  Authentication Architecture:

   Guest Mode (Default):
      • No login required
      • Full tool access
      • No data persistence
      • Session ends when browser closes
      • Perfect for demos and testing

   Registered User Mode:
      • Simple signup (username + password)
      • Secure password hashing (bcrypt)
      • Usage history saved to disk
      • API key generation
      • Personal analytics dashboard

   Authentication Flow:
      1. App loads
      2. Check st.session_state.user
      3. If None:
         • Show 'Continue as Guest' button
         • Show 'Login' form
         • Show 'Sign Up' form
      4. If logged in:
         • Show username + logout button
         • Enable history tracking
         • Show user dashboard

🔐 Security Measures:

   Password Security:
      • bcrypt hashing (industry standard)
      • Automatic salt generation
      • Slow by desi

In [5]:
# ==================================================
# EXERCISE 1.2: IMPLEMENT USER MANAGER CLASS
# ==================================================

print("\n" + "="*80)
print("EXERCISE 1.2: Building User Management System")
print("="*80)

"""
📖 THEORY: User Management Implementation

Class-Based Design:
==================================================

Why Classes:
- Encapsulation (data + methods together)
- Reusability (use across app)
- Maintainability (easier to update)
- State management (users dict)

UserManager Class Structure:
```python
class UserManager:
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.users_file = data_dir / "users.json"
        self.users = self.load_users()
    
    def register_user(self, username, password, email):
        # Hash password, create user, save
    
    def login_user(self, username, password):
        # Verify credentials, return user or None
    
    def get_user(self, username):
        # Retrieve user data
    
    def update_user(self, username, updates):
        # Update user fields
```

Password Hashing with bcrypt:
==================================================

Hashing:
```python
import bcrypt

password = "mypassword"
salt = bcrypt.gensalt()  # Generate salt
hashed = bcrypt.hashpw(password.encode(), salt)
# Returns: b'$2b$12$...' (60 bytes)
```

Verification:
```python
if bcrypt.checkpw(password.encode(), hashed):
    return True  # Password correct
else:
    return False  # Password wrong
```

Why encode()?
- bcrypt needs bytes, not strings
- .encode() converts str → bytes
- Default encoding: UTF-8

JSON Storage:
==================================================

Challenge: bcrypt hashes are bytes, JSON needs strings

Solution: Base64 encoding
```python
import base64

# Save
hash_b64 = base64.b64encode(hashed).decode()
json.dump({'hash': hash_b64}, f)

# Load
hashed = base64.b64decode(hash_b64.encode())
```

User Registration Flow:
==================================================

1. User submits username, password, email
2. Validate:
   - Username not taken
   - Password strength OK
   - Email format valid (optional)
3. Hash password with bcrypt
4. Generate API key
5. Create user object
6. Save to JSON
7. Return success

Login Flow:
==================================================

1. User submits username, password
2. Check username exists
3. Load stored hash
4. Verify password with bcrypt.checkpw()
5. If valid:
   - Update last_login
   - Return user object
6. If invalid:
   - Return None

Error Handling:
==================================================

Common Errors:
- Username already exists
- Username not found
- Wrong password
- Invalid email format
- File write errors

Best Practices:
- Clear error messages
- Don't reveal if username exists (security)
- Generic "Invalid credentials" for login
- Specific errors for registration
"""

print("\n⏱️  Implementing UserManager class...")

# ==================================================
# UserManager Class
# ==================================================

import base64

class UserManager:
    """Manage user accounts with secure password storage."""
    
    def __init__(self, data_dir):
        """
        Initialize UserManager.
        
        Args:
            data_dir: Path to data directory
        """
        self.data_dir = Path(data_dir)
        self.users_file = self.data_dir / "users.json"
        self.users = self.load_users()
    
    def load_users(self):
        """Load users from JSON file."""
        if self.users_file.exists():
            try:
                with open(self.users_file, 'r') as f:
                    return json.load(f)
            except:
                return {}
        return {}
    
    def save_users(self):
        """Save users to JSON file."""
        try:
            with open(self.users_file, 'w') as f:
                json.dump(self.users, f, indent=2)
            return True
        except Exception as e:
            print(f"Error saving users: {e}")
            return False
    
    def hash_password(self, password):
        """
        Hash password with bcrypt.
        
        Args:
            password: Plain text password
        
        Returns:
            str: Base64-encoded hash (for JSON storage)
        """
        hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
        # Convert bytes to base64 string for JSON
        return base64.b64encode(hashed).decode()
    
    def verify_password(self, password, hash_b64):
        """
        Verify password against hash.
        
        Args:
            password: Plain text password
            hash_b64: Base64-encoded hash from storage
        
        Returns:
            bool: True if password matches
        """
        try:
            # Convert base64 string back to bytes
            hashed = base64.b64decode(hash_b64.encode())
            return bcrypt.checkpw(password.encode(), hashed)
        except:
            return False
    
    def generate_api_key(self):
        """
        Generate random API key.
        
        Returns:
            str: API key in format 'sk_...'
        """
        random_bytes = secrets.token_bytes(32)
        key = base64.b64encode(random_bytes).decode()[:40]
        return f"sk_{key}"
    
    def register_user(self, username, password, email=""):
        """
        Register new user.
        
        Args:
            username: Unique username
            password: Plain text password
            email: Optional email
        
        Returns:
            tuple: (success: bool, message: str)
        """
        # Validate username
        if not username or len(username) < 3:
            return False, "Username must be at least 3 characters"
        
        if username in self.users:
            return False, "Username already exists"
        
        # Validate password
        if not password or len(password) < 6:
            return False, "Password must be at least 6 characters"
        
        # Create user
        user = {
            'username': username,
            'password_hash': self.hash_password(password),
            'email': email,
            'created_at': datetime.now().isoformat(),
            'api_key': self.generate_api_key(),
            'usage_count': 0,
            'last_login': None,
            'tools_used': {
                'sentiment': 0,
                'summarizer': 0,
                'fake_news': 0,
                'job_matcher': 0
            }
        }
        
        # Save
        self.users[username] = user
        if self.save_users():
            return True, "Account created successfully!"
        else:
            return False, "Error saving account"
    
    def login_user(self, username, password):
        """
        Authenticate user.
        
        Args:
            username: Username
            password: Plain text password
        
        Returns:
            dict or None: User object if valid, None if invalid
        """
        # Check username exists
        if username not in self.users:
            return None
        
        user = self.users[username]
        
        # Verify password
        if self.verify_password(password, user['password_hash']):
            # Update last login
            user['last_login'] = datetime.now().isoformat()
            self.save_users()
            return user
        
        return None
    
    def get_user(self, username):
        """Get user by username."""
        return self.users.get(username)
    
    def update_user(self, username, updates):
        """
        Update user fields.
        
        Args:
            username: Username
            updates: Dict of fields to update
        
        Returns:
            bool: Success
        """
        if username not in self.users:
            return False
        
        self.users[username].update(updates)
        return self.save_users()
    
    def increment_usage(self, username, tool):
        """
        Increment usage count for tool.
        
        Args:
            username: Username
            tool: Tool name ('sentiment', 'summarizer', etc.)
        """
        if username in self.users:
            user = self.users[username]
            user['usage_count'] += 1
            if tool in user['tools_used']:
                user['tools_used'][tool] += 1
            self.save_users()

print("✅ UserManager class implemented")

# ==================================================
# Test UserManager
# ==================================================

print("\n📊 UserManager Features:")

print("\n   Core Methods:")
print("      • __init__(data_dir) - Initialize with data directory")
print("      • load_users() - Load from JSON")
print("      • save_users() - Save to JSON")
print("      • hash_password(password) - Bcrypt hash to base64")
print("      • verify_password(password, hash) - Check password")
print("      • generate_api_key() - Random 'sk_...' key")

print("\n   User Operations:")
print("      • register_user(username, password, email)")
print("        Returns: (success, message)")
print("      • login_user(username, password)")
print("        Returns: user object or None")
print("      • get_user(username)")
print("        Returns: user object")
print("      • update_user(username, updates)")
print("        Returns: success boolean")
print("      • increment_usage(username, tool)")
print("        Updates usage counters")

print("\n🔐 Security Features:")
print("   • Bcrypt password hashing (12 rounds)")
print("   • Automatic salt generation")
print("   • Base64 encoding for JSON storage")
print("   • Generic login errors (don't reveal if user exists)")
print("   • API keys: 32-byte random secrets")

print("\n💾 Data Persistence:")
print("   • Storage: JSON file (user_data/users.json)")
print("   • Format: {username: user_object}")
print("   • Automatic save after modifications")
print("   • Error handling for file operations")

print("\n✅ Exercise 1.2 Complete!")
print("="*80)


EXERCISE 1.2: Building User Management System

⏱️  Implementing UserManager class...
✅ UserManager class implemented

📊 UserManager Features:

   Core Methods:
      • __init__(data_dir) - Initialize with data directory
      • load_users() - Load from JSON
      • save_users() - Save to JSON
      • hash_password(password) - Bcrypt hash to base64
      • verify_password(password, hash) - Check password
      • generate_api_key() - Random 'sk_...' key

   User Operations:
      • register_user(username, password, email)
        Returns: (success, message)
      • login_user(username, password)
        Returns: user object or None
      • get_user(username)
        Returns: user object
      • update_user(username, updates)
        Returns: success boolean
      • increment_usage(username, tool)
        Updates usage counters

🔐 Security Features:
   • Bcrypt password hashing (12 rounds)
   • Automatic salt generation
   • Base64 encoding for JSON storage
   • Generic login errors (don

In [6]:
# ==================================================
# EXERCISE 1.3: BUILD AUTHENTICATION UI
# ==================================================

print("\n" + "="*80)
print("EXERCISE 1.3: Creating Authentication Interface")
print("="*80)

"""
📖 THEORY: Authentication UI Design

UI Components Needed:
==================================================

1. Guest Mode Button:
   - Continue without account
   - Instant access
   - No persistence

2. Login Form:
   - Username input
   - Password input
   - Submit button
   - Error messages

3. Signup Form:
   - Username input
   - Email input (optional)
   - Password input
   - Confirm password
   - Submit button
   - Validation messages

4. User Info Display:
   - Username
   - Logout button
   - Usage stats
   - API key (hidden/show)

Streamlit Form Components:
==================================================

st.form():
```python
with st.form("login_form"):
    username = st.text_input("Username")
    password = st.text_input("Password", type="password")
    submitted = st.form_submit_button("Login")

if submitted:
    # Process login
```

Why Forms:
- Batches inputs (no rerun per keystroke)
- Enter key submits
- Better UX

Password Input:
```python
password = st.text_input("Password", type="password")
# Shows ••••• instead of text
```

Session State for Auth:
==================================================

Pattern:
```python
# Initialize
if 'user' not in st.session_state:
    st.session_state.user = None

# Check login status
if st.session_state.user:
    st.write(f"Welcome, {st.session_state.user['username']}!")
else:
    # Show login UI
```

UI Flow:
==================================================

State 1: Not Logged In
┌─────────────────────────┐
│  TextAI Studio          │
├─────────────────────────┤
│  [Continue as Guest]    │
│                         │
│  OR                     │
│                         │
│  Login:                 │
│    Username: [____]     │
│    Password: [••••]     │
│    [Login Button]       │
│                         │
│  Don't have account?    │
│  [Sign Up]              │
└─────────────────────────┘

State 2: Logged In
┌─────────────────────────┐
│  TextAI Studio          │
│  User: alice [Logout]   │
├─────────────────────────┤
│  [Tool Interface]       │
│  [Usage Stats]          │
│  [API Key Management]   │
└─────────────────────────┘

Tabs for Login/Signup:
==================================================

Better UX:
```python
tab1, tab2 = st.tabs(["Login", "Sign Up"])

with tab1:
    # Login form

with tab2:
    # Signup form
```

Cleaner than side-by-side forms.

Validation:
==================================================

Client-Side (Before Submit):
- Username length (3+ chars)
- Password length (6+ chars)
- Email format (basic check)
- Password confirmation match

Server-Side (UserManager):
- Username unique
- All validation checks
- Database constraints

Error Display:
```python
if error:
    st.error(f"❌ {error_message}")

if success:
    st.success(f"✅ {success_message}")
```

Password Confirmation:
==================================================

Pattern:
```python
password = st.text_input("Password", type="password")
confirm = st.text_input("Confirm Password", type="password")

if password != confirm:
    st.error("Passwords don't match!")
```
"""

print("\n⏱️  Implementing authentication UI functions...")

# ==================================================
# Authentication UI Functions
# ==================================================

def show_auth_ui(user_manager):
    """
    Display authentication UI.
    
    Args:
        user_manager: UserManager instance
    """
    st.markdown("### 🔐 Authentication")
    
    # Guest mode button
    if st.button("👤 Continue as Guest", use_container_width=True, type="primary"):
        st.session_state.user = {'username': 'Guest', 'is_guest': True}
        st.rerun()
    
    st.markdown("---")
    st.markdown("**OR**")
    st.markdown("---")
    
    # Login / Signup tabs
    tab1, tab2 = st.tabs(["🔑 Login", "📝 Sign Up"])
    
    # Login Tab
    with tab1:
        with st.form("login_form"):
            st.markdown("#### Login to Your Account")
            
            username = st.text_input(
                "Username",
                placeholder="Enter your username"
            )
            
            password = st.text_input(
                "Password",
                type="password",
                placeholder="Enter your password"
            )
            
            submitted = st.form_submit_button("Login", use_container_width=True)
        
        if submitted:
            if not username or not password:
                st.error("❌ Please enter both username and password")
            else:
                user = user_manager.login_user(username, password)
                
                if user:
                    st.session_state.user = user
                    st.success(f"✅ Welcome back, {username}!")
                    time.sleep(1)
                    st.rerun()
                else:
                    st.error("❌ Invalid username or password")
    
    # Signup Tab
    with tab2:
        with st.form("signup_form"):
            st.markdown("#### Create New Account")
            
            new_username = st.text_input(
                "Username",
                placeholder="Choose a username (3+ characters)"
            )
            
            email = st.text_input(
                "Email (optional)",
                placeholder="your.email@example.com"
            )
            
            new_password = st.text_input(
                "Password",
                type="password",
                placeholder="Choose a password (6+ characters)"
            )
            
            confirm_password = st.text_input(
                "Confirm Password",
                type="password",
                placeholder="Re-enter your password"
            )
            
            signup_submitted = st.form_submit_button("Sign Up", use_container_width=True)
        
        if signup_submitted:
            # Validation
            if not new_username or not new_password:
                st.error("❌ Username and password are required")
            elif len(new_username) < 3:
                st.error("❌ Username must be at least 3 characters")
            elif len(new_password) < 6:
                st.error("❌ Password must be at least 6 characters")
            elif new_password != confirm_password:
                st.error("❌ Passwords don't match")
            else:
                success, message = user_manager.register_user(
                    new_username, 
                    new_password, 
                    email
                )
                
                if success:
                    st.success(f"✅ {message}")
                    st.info("💡 You can now login with your credentials")
                else:
                    st.error(f"❌ {message}")

print("✅ show_auth_ui() implemented")

# ==================================================
# User Info Display
# ==================================================

def show_user_info(user):
    """
    Display logged-in user info.
    
    Args:
        user: User object from session state
    """
    if user.get('is_guest'):
        st.sidebar.markdown("### 👤 Guest Mode")
        st.sidebar.info("💡 Create an account to save history and get API keys")
        return
    
    st.sidebar.markdown(f"### 👤 {user['username']}")
    
    # Logout button
    if st.sidebar.button("🚪 Logout", use_container_width=True):
        st.session_state.user = None
        st.rerun()
    
    # Usage stats
    with st.sidebar.expander("📊 Usage Stats"):
        st.metric("Total Queries", user.get('usage_count', 0))
        
        tools_used = user.get('tools_used', {})
        if tools_used:
            st.markdown("**By Tool:**")
            for tool, count in tools_used.items():
                st.write(f"• {tool.title()}: {count}")
    
    # API key
    with st.sidebar.expander("🔑 API Key"):
        api_key = user.get('api_key', 'Not generated')
        
        # Show/hide toggle
        if 'show_api_key' not in st.session_state:
            st.session_state.show_api_key = False
        
        if st.session_state.show_api_key:
            st.code(api_key, language=None)
            if st.button("🙈 Hide"):
                st.session_state.show_api_key = False
                st.rerun()
        else:
            st.text("•" * 40)
            if st.button("👁️ Show"):
                st.session_state.show_api_key = True
                st.rerun()
    
    # Account info
    with st.sidebar.expander("ℹ️ Account Info"):
        created = user.get('created_at', 'Unknown')
        if created != 'Unknown':
            created = datetime.fromisoformat(created).strftime("%Y-%m-%d")
        st.write(f"**Created:** {created}")
        
        last_login = user.get('last_login')
        if last_login:
            last_login = datetime.fromisoformat(last_login).strftime("%Y-%m-%d %H:%M")
            st.write(f"**Last Login:** {last_login}")

print("✅ show_user_info() implemented")

# ==================================================
# Session Initialization
# ==================================================

def init_auth_session():
    """Initialize authentication session state."""
    if 'user' not in st.session_state:
        st.session_state.user = None

print("✅ init_auth_session() implemented")

# ==================================================
# Helper Functions
# ==================================================

def is_logged_in():
    """Check if user is logged in."""
    return st.session_state.get('user') is not None

def get_current_user():
    """Get current user from session."""
    return st.session_state.get('user')

def is_guest():
    """Check if current user is guest."""
    user = get_current_user()
    return user and user.get('is_guest', False)

print("✅ Helper functions implemented")

print("\n📊 Authentication UI Summary:")

print("\n   Functions Created:")
print("      • show_auth_ui(user_manager)")
print("        - Guest mode button")
print("        - Login tab (form)")
print("        - Signup tab (form)")
print("      • show_user_info(user)")
print("        - Username display")
print("        - Logout button")
print("        - Usage stats")
print("        - API key (show/hide)")
print("        - Account info")
print("      • init_auth_session()")
print("        - Initialize session state")
print("      • Helper functions:")
print("        - is_logged_in()")
print("        - get_current_user()")
print("        - is_guest()")

print("\n🎨 UI Features:")
print("   • Guest mode for instant access")
print("   • Tabbed login/signup interface")
print("   • Form-based inputs (Enter to submit)")
print("   • Password masking (type='password')")
print("   • Password confirmation")
print("   • Client-side validation")
print("   • Clear error/success messages")
print("   • Sidebar user info")
print("   • API key show/hide toggle")
print("   • Usage statistics")

print("\n✅ Exercise 1.3 Complete!")
print("="*80)


EXERCISE 1.3: Creating Authentication Interface

⏱️  Implementing authentication UI functions...
✅ show_auth_ui() implemented
✅ show_user_info() implemented
✅ init_auth_session() implemented
✅ Helper functions implemented

📊 Authentication UI Summary:

   Functions Created:
      • show_auth_ui(user_manager)
        - Guest mode button
        - Login tab (form)
        - Signup tab (form)
      • show_user_info(user)
        - Username display
        - Logout button
        - Usage stats
        - API key (show/hide)
        - Account info
      • init_auth_session()
        - Initialize session state
      • Helper functions:
        - is_logged_in()
        - get_current_user()
        - is_guest()

🎨 UI Features:
   • Guest mode for instant access
   • Tabbed login/signup interface
   • Form-based inputs (Enter to submit)
   • Password masking (type='password')
   • Password confirmation
   • Client-side validation
   • Clear error/success messages
   • Sidebar user info
   • API

In [7]:
# ==================================================
# EXERCISE 1.4: TEST AUTHENTICATION SYSTEM
# ==================================================

print("\n" + "="*80)
print("EXERCISE 1.4: Testing Authentication Components")
print("="*80)

"""
📖 THEORY: Testing Authentication Systems

Testing Strategy:
==================================================

Unit Tests:
- Test each function individually
- Mock dependencies
- Verify outputs

Integration Tests:
- Test complete flows
- End-to-end scenarios
- Real interactions

Manual Tests:
- UI testing
- User experience
- Edge cases

Test Cases for Authentication:
==================================================

UserManager Tests:
1. Password hashing
   - Hash produces different output each time (salt)
   - Verification works correctly
   - Base64 encoding/decoding

2. User registration
   - Valid user created
   - Duplicate username rejected
   - Short password rejected
   - Short username rejected

3. User login
   - Valid credentials succeed
   - Invalid username fails
   - Invalid password fails
   - Last login updated

4. Usage tracking
   - Increment works
   - Multiple tools tracked
   - Counts persist

UI Tests:
1. Guest mode
   - Button works
   - Session updated
   - No persistence

2. Login
   - Form submission
   - Validation
   - Error messages
   - Success flow

3. Signup
   - Form submission
   - Password confirmation
   - Validation
   - Success flow

4. User info display
   - Stats shown
   - API key hide/show
   - Logout works

Edge Cases:
==================================================

- Empty inputs
- Special characters in username
- Very long inputs
- SQL injection attempts (N/A for JSON)
- Concurrent logins
- Session expiry
- File corruption
- Disk full
"""

print("\n⏱️  Running authentication tests...")

print("\n🧪 Test Suite:")

# ==================================================
# Test 1: UserManager Initialization
# ==================================================

print("\n   Test 1: UserManager Initialization")
try:
    test_manager = UserManager(USER_DATA_DIR)
    print("      ✅ UserManager initialized")
    print(f"      ✅ Users loaded: {len(test_manager.users)} users")
except Exception as e:
    print(f"      ❌ Initialization failed: {e}")

# ==================================================
# Test 2: Password Hashing
# ==================================================

print("\n   Test 2: Password Hashing & Verification")
try:
    test_password = "test123456"
    
    # Hash password
    hash1 = test_manager.hash_password(test_password)
    hash2 = test_manager.hash_password(test_password)
    
    # Hashes should be different (due to salt)
    if hash1 != hash2:
        print("      ✅ Each hash is unique (salt working)")
    else:
        print("      ⚠️ Hashes are identical (salt might not be working)")
    
    # Verification should work for both
    if test_manager.verify_password(test_password, hash1):
        print("      ✅ Password verification works (hash 1)")
    else:
        print("      ❌ Password verification failed (hash 1)")
    
    if test_manager.verify_password(test_password, hash2):
        print("      ✅ Password verification works (hash 2)")
    else:
        print("      ❌ Password verification failed (hash 2)")
    
    # Wrong password should fail
    if not test_manager.verify_password("wrong", hash1):
        print("      ✅ Wrong password correctly rejected")
    else:
        print("      ❌ Wrong password incorrectly accepted")

except Exception as e:
    print(f"      ❌ Password test failed: {e}")

# ==================================================
# Test 3: API Key Generation
# ==================================================

print("\n   Test 3: API Key Generation")
try:
    key1 = test_manager.generate_api_key()
    key2 = test_manager.generate_api_key()
    
    print(f"      ✅ Generated key 1: {key1[:20]}...")
    print(f"      ✅ Generated key 2: {key2[:20]}...")
    
    if key1 != key2:
        print("      ✅ Keys are unique")
    else:
        print("      ❌ Keys are identical")
    
    if key1.startswith("sk_") and len(key1) > 40:
        print("      ✅ Key format correct (sk_...)")
    else:
        print("      ❌ Key format incorrect")

except Exception as e:
    print(f"      ❌ API key test failed: {e}")

# ==================================================
# Test 4: User Registration
# ==================================================

print("\n   Test 4: User Registration")

# Valid registration
success, msg = test_manager.register_user("test_user_123", "password123", "test@example.com")
if success:
    print(f"      ✅ Valid registration succeeded: {msg}")
else:
    print(f"      ⚠️ Registration result: {msg}")

# Duplicate username
success, msg = test_manager.register_user("test_user_123", "password456", "test2@example.com")
if not success and "already exists" in msg:
    print(f"      ✅ Duplicate username rejected: {msg}")
else:
    print(f"      ❌ Duplicate username not properly rejected: {msg}")

# Short username
success, msg = test_manager.register_user("ab", "password123", "test3@example.com")
if not success and "3 characters" in msg:
    print(f"      ✅ Short username rejected: {msg}")
else:
    print(f"      ❌ Short username not properly rejected: {msg}")

# Short password
success, msg = test_manager.register_user("valid_user", "12345", "test4@example.com")
if not success and "6 characters" in msg:
    print(f"      ✅ Short password rejected: {msg}")
else:
    print(f"      ❌ Short password not properly rejected: {msg}")

# ==================================================
# Test 5: User Login
# ==================================================

print("\n   Test 5: User Login")

# Valid login
user = test_manager.login_user("test_user_123", "password123")
if user:
    print(f"      ✅ Valid login succeeded for: {user['username']}")
else:
    print("      ❌ Valid login failed")

# Invalid username
user = test_manager.login_user("nonexistent", "password123")
if user is None:
    print("      ✅ Invalid username rejected")
else:
    print("      ❌ Invalid username not properly rejected")

# Invalid password
user = test_manager.login_user("test_user_123", "wrongpassword")
if user is None:
    print("      ✅ Invalid password rejected")
else:
    print("      ❌ Invalid password not properly rejected")

# ==================================================
# Test Summary
# ==================================================

print("\n📊 Test Summary:")
print("   • UserManager: Tested ✅")
print("   • Password hashing: Tested ✅")
print("   • API key generation: Tested ✅")
print("   • User registration: Tested ✅")
print("   • User login: Tested ✅")
print("   • Validation: Tested ✅")

print("\n💡 Manual UI Tests (Run in Streamlit app):")
print("   1. Guest mode button")
print("   2. Login form submission")
print("   3. Signup form submission")
print("   4. Password confirmation")
print("   5. User info display")
print("   6. API key show/hide")
print("   7. Logout button")
print("   8. Usage stats display")

print("\n🔒 Security Tests Passed:")
print("   ✅ Passwords hashed with bcrypt")
print("   ✅ Unique salts per password")
print("   ✅ Password verification secure")
print("   ✅ API keys randomly generated")
print("   ✅ Validation prevents bad inputs")
print("   ✅ Generic login errors (security)")

print("\n✅ Exercise 1.4 Complete!")
print("="*80)


EXERCISE 1.4: Testing Authentication Components

⏱️  Running authentication tests...

🧪 Test Suite:

   Test 1: UserManager Initialization
      ✅ UserManager initialized
      ✅ Users loaded: 0 users

   Test 2: Password Hashing & Verification
      ✅ Each hash is unique (salt working)
      ✅ Password verification works (hash 1)
      ✅ Password verification works (hash 2)
      ✅ Wrong password correctly rejected

   Test 3: API Key Generation
      ✅ Generated key 1: sk_1Js8FLrjoYHzpzg/I...
      ✅ Generated key 2: sk_MfJtYCJzS2V6MeRAB...
      ✅ Keys are unique
      ✅ Key format correct (sk_...)

   Test 4: User Registration
      ✅ Valid registration succeeded: Account created successfully!
      ✅ Duplicate username rejected: Username already exists
      ✅ Short username rejected: Username must be at least 3 characters
      ✅ Short password rejected: Password must be at least 6 characters

   Test 5: User Login
      ✅ Valid login succeeded for: test_user_123
      ✅ Invalid

In [8]:
print("\n" + "="*80)
print("📊 PART 2: USAGE HISTORY & SESSION MANAGEMENT")
print("="*80)


📊 PART 2: USAGE HISTORY & SESSION MANAGEMENT


In [11]:
# ==================================================
# EXERCISE 2.1: DESIGN USAGE HISTORY SYSTEM
# ==================================================

print("\n" + "="*80)
print("EXERCISE 2.1: Planning Usage History Architecture")
print("="*80)

"""
📖 THEORY: Usage History & Analytics

What is Usage History?
==================================================

Purpose:
- Track user interactions
- Show past analyses
- Enable re-running queries
- Provide analytics
- Improve user experience

Benefits:
1. User: See what they've done
2. Developer: Understand usage patterns
3. Product: Feature usage data
4. Support: Debug issues

Data to Track:
==================================================

Per Query:
- Timestamp (when)
- Tool used (sentiment, summary, etc.)
- Input text (or hash for privacy)
- Result (output)
- Processing time (performance)
- Success/error status

Per User:
- Total queries
- Queries by tool
- Last login time
- Account creation date
- API key usage

Storage Options:
==================================================

1. In-Memory (Session State):
   - Fast
   - Lost on browser close
   - Good for: Current session

2. Local Files (JSON/CSV):
   - Persistent
   - Easy to implement
   - Good for: Demo/portfolio

3. Database (SQLite/PostgreSQL):
   - Scalable
   - Query-able
   - Good for: Production

4. Cloud (Firebase/Supabase):
   - Managed
   - Scalable
   - Good for: Deployed apps

Our Approach: Local JSON Files
==================================================

Structure:user_data/
├── history/
│   ├── alice.json
│   ├── bob.json
│   └── guest_session_abc123.json

File Format:
```json{
"username": "alice",
"history": [
{
"id": "uuid-1234",
"timestamp": "2024-12-26T10:30:00",
"tool": "sentiment",
"input": "This is great!",
"result": {
"sentiment": "Positive",
"confidence": 95.2
},
"processing_time_ms": 127,
"success": true
},
...
]
}

History Entry Structure:
==================================================
```pythonentry = {
'id': str(uuid.uuid4()),  # Unique ID
'timestamp': datetime.now().isoformat(),
'tool': 'sentiment',  # Tool name
'input': 'Text here...',  # Or hash
'result': {...},  # Tool output
'processing_time_ms': 127,
'success': True,
'error': None  # If failed
}

Privacy Considerations:
==================================================

Options:
1. Store Full Input:
   - Pro: Can re-run
   - Con: Privacy concerns

2. Store Hash Only:
   - Pro: Privacy-friendly
   - Con: Can't see original

3. Store Summary:
   - Pro: Balance
   - Con: Lossy

Our Approach: Store full input
- It's their data
- Local storage (not cloud)
- They can delete anytime

History Display:
==================================================

UI Components:
1. History List:
   - Reverse chronological
   - Show: Time, Tool, Preview
   - Click to expand

2. Entry Detail:
   - Full input
   - Full result
   - Timestamp
   - Processing time

3. Actions:
   - Re-run query
   - Copy input
   - Delete entry
   - Export history

Filters:
- By tool
- By date range
- By success/error
- Search in input

Analytics Dashboard:
==================================================

Metrics to Show:
- Total queries
- Queries per tool
- Average processing time
- Success rate
- Activity over time

Visualizations:
- Bar chart: Queries by tool
- Line chart: Activity over time
- Pie chart: Tool distribution
- Table: Recent queries

Implementation Strategy:
==================================================

Components:
1. HistoryManager class
   - add_entry()
   - get_history()
   - get_by_tool()
   - get_by_date()
   - delete_entry()
   - clear_history()

2. History UI
   - display_history()
   - show_entry_detail()
   - history_filters()

3. Analytics UI
   - show_analytics_dashboard()
   - generate_charts()

Session vs Persistent:
==================================================

Guest Mode:
- History in session state
- Lost on browser close
- No file storage

Logged In:
- History saved to JSON
- Persists across sessions
- Can be exported

Why This Matters:
==================================================

User Experience:
- Don't repeat queries
- Learn from past
- Track progress

Analytics:
- Which tools popular
- Where users stuck
- Performance issues

Portfolio:
- Shows full-stack thinking
- Data persistence
- User-centric design
"""

print("\n⏱️  Designing usage history system...")

print("\n🏗️  History System Architecture:")

print("\n   Data Structure:")
print("      History Entry:")
print("         • id: Unique UUID")
print("         • timestamp: ISO format datetime")
print("         • tool: Tool name (sentiment, etc.)")
print("         • input: User input text")
print("         • result: Tool output (dict)")
print("         • processing_time_ms: Latency")
print("         • success: Boolean")
print("         • error: Error message if failed")

print("\n   Storage Strategy:")
print("      Guest Mode:")
print("         • Session state only")
print("         • No file persistence")
print("         • Lost on close")
print("      Logged In:")
print("         • JSON file per user")
print("         • Location: user_data/history/{username}.json")
print("         • Persistent across sessions")

print("\n   File Structure:")
print("      user_data/")
print("         history/")
print("            alice.json")
print("            bob.json")
print("            guest_session_xyz.json")

print("\n📊 History Features:")

print("\n   1. History Tracking:")
print("      • Automatic logging of all queries")
print("      • Tool identification")
print("      • Success/error status")
print("      • Performance metrics")

print("\n   2. History Display:")
print("      • Reverse chronological list")
print("      • Expandable entries")
print("      • Preview of input/output")
print("      • Timestamp formatting")

print("\n   3. Filters & Search:")
print("      • Filter by tool")
print("      • Filter by date range")
print("      • Filter by success/error")
print("      • Search in input text")

print("\n   4. Actions:")
print("      • Re-run query (copy input)")
print("      • Delete single entry")
print("      • Clear all history")
print("      • Export to CSV/JSON")

print("\n   5. Analytics:")
print("      • Total query count")
print("      • Queries per tool")
print("      • Success rate")
print("      • Average processing time")
print("      • Activity timeline")

print("\n🎨 UI Components:")

print("\n   History Sidebar:")
print("      • Recent queries (5 most recent)")
print("      • Quick stats")
print("      • 'View All' button")

print("\n   Full History Page:")
print("      • All entries with pagination")
print("      • Filters dropdown")
print("      • Search bar")
print("      • Export button")

print("\n   Analytics Dashboard:")
print("      • Metric cards")
print("      • Tool usage chart (bar)")
print("      • Activity timeline (line)")
print("      • Success rate (metric)")

print("\n💾 Data Persistence:")

print("\n   JSON Format:")
code = '''
{
  "username": "alice",
  "history": [
    {
      "id": "abc-123",
      "timestamp": "2024-12-26T10:30:00",
      "tool": "sentiment",
      "input": "This product is amazing!",
      "result": {
        "sentiment": "Positive",
        "confidence": 95.2,
        "scores": {"negative": 4.8, "positive": 95.2}
      },
      "processing_time_ms": 127,
      "success": true,
      "error": null
    }
  ]
}
'''
print(code)

print("\n   File Operations:")
print("      • Load: Read JSON on user login")
print("      • Save: Write after each query")
print("      • Append: Add new entries")
print("      • Delete: Remove specific entries")

print("\n🔒 Privacy & Security:")

print("\n   Data Storage:")
print("      • Local files only (not cloud)")
print("      • User owns their data")
print("      • Can delete anytime")
print("      • Can export anytime")

print("\n   Guest Mode:")
print("      • No persistence")
print("      • Session only")
print("      • Privacy-friendly")

print("\n✅ Exercise 2.1 Complete!")
print("="*80)


EXERCISE 2.1: Planning Usage History Architecture

⏱️  Designing usage history system...

🏗️  History System Architecture:

   Data Structure:
      History Entry:
         • id: Unique UUID
         • timestamp: ISO format datetime
         • tool: Tool name (sentiment, etc.)
         • input: User input text
         • result: Tool output (dict)
         • processing_time_ms: Latency
         • success: Boolean
         • error: Error message if failed

   Storage Strategy:
      Guest Mode:
         • Session state only
         • No file persistence
         • Lost on close
      Logged In:
         • JSON file per user
         • Location: user_data/history/{username}.json
         • Persistent across sessions

   File Structure:
      user_data/
         history/
            alice.json
            bob.json
            guest_session_xyz.json

📊 History Features:

   1. History Tracking:
      • Automatic logging of all queries
      • Tool identification
      • Success/error sta

In [12]:
# ==================================================
# EXERCISE 2.2: IMPLEMENT HISTORY MANAGER
# ==================================================

print("\n" + "="*80)
print("EXERCISE 2.2: Building History Management System")
print("="*80)

"""
📖 THEORY: History Management Implementation

Class Design:
==================================================

HistoryManager:
- Manages history for all users
- File operations (load/save)
- Add/delete/query entries
- Analytics calculations

Methods:
- add_entry(username, entry)
- get_history(username, limit=None)
- get_by_tool(username, tool)
- get_by_date(username, start, end)
- delete_entry(username, entry_id)
- clear_history(username)
- get_analytics(username)

File Management:
==================================================

Per-User Files:
- One JSON file per user
- Lazy loading (load on demand)
- Auto-save after modifications

Format:
{
  "username": "alice",
  "history": [...]
}

Append Pattern:
1. Load existing history
2. Append new entry
3. Save back to file

Optimization:
- Keep in memory while session active
- Only write when changed
- Batch writes if needed

Entry ID Generation:
==================================================

UUID:
```pythonimport uuid
entry_id = str(uuid.uuid4())
Example: "123e4567-e89b-12d3-a456-426614174000"

Why UUID:
- Globally unique
- No collisions
- Easy to generate
- Standard format

Query Operations:
==================================================

Filter by Tool:
```pythonentries = [e for e in history if e['tool'] == 'sentiment']

Filter by Date:
```pythonentries = [
e for e in history
if start <= e['timestamp'] <= end
]

Sort by Time:
```pythonentries.sort(key=lambda e: e['timestamp'], reverse=True)

Analytics Calculations:
==================================================

Total Queries:
len(history)

By Tool:
```pythonfrom collections import Counter
tools = [e['tool'] for e in history]
counts = Counter(tools)

Success Rate:
```pythonsuccesses = sum(1 for e in history if e['success'])
rate = (successes / len(history)) * 100

Average Time:
```pythontimes = [e['processing_time_ms'] for e in history]
avg = sum(times) / len(times)

Error Handling:
==================================================

Common Issues:
- File not found (first time)
- JSON parse error (corrupted)
- Disk full (write failure)
- Invalid entry format

Solutions:
- Create file if missing
- Handle parse errors gracefully
- Catch write exceptions
- Validate entry structure
"""

print("\n⏱️  Implementing HistoryManager class...")

# ==================================================
# HistoryManager Class
# ==================================================

class HistoryManager:
    """Manage usage history for users."""
    
    def __init__(self, history_dir):
        """
        Initialize HistoryManager.
        
        Args:
            history_dir: Path to history directory
        """
        self.history_dir = Path(history_dir)
        self.history_dir.mkdir(exist_ok=True)
    
    def _get_history_file(self, username):
        """Get history file path for user."""
        return self.history_dir / f"{username}.json"
    
    def load_history(self, username):
        """
        Load history for user.
        
        Args:
            username: Username
        
        Returns:
            list: History entries
        """
        history_file = self._get_history_file(username)
        
        if history_file.exists():
            try:
                with open(history_file, 'r') as f:
                    data = json.load(f)
                    return data.get('history', [])
            except:
                return []
        
        return []
    
    def save_history(self, username, history):
        """
        Save history for user.
        
        Args:
            username: Username
            history: List of history entries
        
        Returns:
            bool: Success
        """
        history_file = self._get_history_file(username)
        
        try:
            data = {
                'username': username,
                'history': history
            }
            
            with open(history_file, 'w') as f:
                json.dump(data, f, indent=2)
            
            return True
        except Exception as e:
            print(f"Error saving history: {e}")
            return False
    
    def add_entry(self, username, tool, input_text, result, processing_time_ms, success=True, error=None):
        """
        Add history entry.
        
        Args:
            username: Username
            tool: Tool name
            input_text: Input text
            result: Tool result (dict)
            processing_time_ms: Processing time
            success: Success boolean
            error: Error message if failed
        
        Returns:
            str: Entry ID
        """
        # Create entry
        entry = {
            'id': str(uuid.uuid4()),
            'timestamp': datetime.now().isoformat(),
            'tool': tool,
            'input': input_text,
            'result': result,
            'processing_time_ms': processing_time_ms,
            'success': success,
            'error': error
        }
        
        # Load existing history
        history = self.load_history(username)
        
        # Append new entry
        history.append(entry)
        
        # Save
        self.save_history(username, history)
        
        return entry['id']
    
    def get_history(self, username, limit=None):
        """
        Get history for user.
        
        Args:
            username: Username
            limit: Max entries to return (None = all)
        
        Returns:
            list: History entries (reverse chronological)
        """
        history = self.load_history(username)
        
        # Sort by timestamp (newest first)
        history.sort(key=lambda e: e['timestamp'], reverse=True)
        
        if limit:
            return history[:limit]
        
        return history
    
    def get_by_tool(self, username, tool):
        """
        Get history filtered by tool.
        
        Args:
            username: Username
            tool: Tool name
        
        Returns:
            list: Filtered history
        """
        history = self.load_history(username)
        return [e for e in history if e['tool'] == tool]
    
    def get_by_date(self, username, start_date=None, end_date=None):
        """
        Get history filtered by date range.
        
        Args:
            username: Username
            start_date: Start datetime (ISO string)
            end_date: End datetime (ISO string)
        
        Returns:
            list: Filtered history
        """
        history = self.load_history(username)
        
        filtered = history
        
        if start_date:
            filtered = [e for e in filtered if e['timestamp'] >= start_date]
        
        if end_date:
            filtered = [e for e in filtered if e['timestamp'] <= end_date]
        
        return filtered
    
    def delete_entry(self, username, entry_id):
        """
        Delete specific entry.
        
        Args:
            username: Username
            entry_id: Entry ID to delete
        
        Returns:
            bool: Success
        """
        history = self.load_history(username)
        
        # Filter out entry
        history = [e for e in history if e['id'] != entry_id]
        
        return self.save_history(username, history)
    
    def clear_history(self, username):
        """
        Clear all history for user.
        
        Args:
            username: Username
        
        Returns:
            bool: Success
        """
        return self.save_history(username, [])
    
    def get_analytics(self, username):
        """
        Calculate analytics for user.
        
        Args:
            username: Username
        
        Returns:
            dict: Analytics data
        """
        history = self.load_history(username)
        
        if not history:
            return {
                'total_queries': 0,
                'by_tool': {},
                'success_rate': 0,
                'avg_processing_time': 0
            }
        
        # Total queries
        total = len(history)
        
        # By tool
        from collections import Counter
        tools = [e['tool'] for e in history]
        by_tool = dict(Counter(tools))
        
        # Success rate
        successes = sum(1 for e in history if e['success'])
        success_rate = (successes / total) * 100
        
        # Average processing time
        times = [e['processing_time_ms'] for e in history if 'processing_time_ms' in e]
        avg_time = sum(times) / len(times) if times else 0
        
        return {
            'total_queries': total,
            'by_tool': by_tool,
            'success_rate': success_rate,
            'avg_processing_time': avg_time
        }

print("✅ HistoryManager class implemented")

# ==================================================
# Test HistoryManager
# ==================================================

print("\n📊 HistoryManager Features:")

print("\n   Core Methods:")
print("      • load_history(username)")
print("        Returns: List of entries")
print("      • save_history(username, history)")
print("        Returns: Success boolean")
print("      • add_entry(username, tool, input, result, ...)")
print("        Returns: Entry ID")

print("\n   Query Methods:")
print("      • get_history(username, limit=None)")
print("        Returns: All entries (reverse chronological)")
print("      • get_by_tool(username, tool)")
print("        Returns: Filtered by tool")
print("      • get_by_date(username, start, end)")
print("        Returns: Filtered by date range")

print("\n   Modification Methods:")
print("      • delete_entry(username, entry_id)")
print("        Returns: Success boolean")
print("      • clear_history(username)")
print("        Returns: Success boolean")

print("\n   Analytics:")
print("      • get_analytics(username)")
print("        Returns: {total, by_tool, success_rate, avg_time}")

print("\n💾 Data Persistence:")
print("   • Storage: JSON files in user_data/history/")
print("   • Format: {username, history: [entries]}")
print("   • Auto-save after modifications")
print("   • Lazy loading (load on demand)")

print("\n🔍 Query Operations:")
print("   • Sort: Reverse chronological (newest first)")
print("   • Filter: By tool, by date range")
print("   • Limit: Return top N entries")
print("   • Delete: By entry ID")

print("\n✅ Exercise 2.2 Complete!")
print("="*80)


EXERCISE 2.2: Building History Management System

⏱️  Implementing HistoryManager class...
✅ HistoryManager class implemented

📊 HistoryManager Features:

   Core Methods:
      • load_history(username)
        Returns: List of entries
      • save_history(username, history)
        Returns: Success boolean
      • add_entry(username, tool, input, result, ...)
        Returns: Entry ID

   Query Methods:
      • get_history(username, limit=None)
        Returns: All entries (reverse chronological)
      • get_by_tool(username, tool)
        Returns: Filtered by tool
      • get_by_date(username, start, end)
        Returns: Filtered by date range

   Modification Methods:
      • delete_entry(username, entry_id)
        Returns: Success boolean
      • clear_history(username)
        Returns: Success boolean

   Analytics:
      • get_analytics(username)
        Returns: {total, by_tool, success_rate, avg_time}

💾 Data Persistence:
   • Storage: JSON files in user_data/history/
   • F

In [13]:
# ==================================================
# EXERCISE 2.3: BUILD HISTORY UI COMPONENTS
# ==================================================

print("\n" + "="*80)
print("EXERCISE 2.3: Creating History Display Interface")
print("="*80)

"""
📖 THEORY: History UI Design

UI Components:
==================================================

1. Recent History Sidebar:
   - Show 5 most recent
   - Tool icon + preview
   - Click to expand
   - "View All" link

2. Full History Page:
   - All entries with pagination
   - Filters (tool, date)
   - Search bar
   - Export button
   - Delete actions

3. Entry Detail:
   - Full input text
   - Full result
   - Timestamp
   - Processing time
   - Re-run button

Display Patterns:
==================================================

Expandable Cards:
```pythonwith st.expander(f"{tool} - {timestamp}"):
st.write("Input:", input)
st.write("Result:", result)

Tabular Display:
```pythondf = pd.DataFrame(history)
st.dataframe(df[['timestamp', 'tool', 'success']])

Metric Cards:
```pythoncol1, col2, col3 = st.columns(3)
col1.metric("Total", total)
col2.metric("Success Rate", f"{rate}%")
col3.metric("Avg Time", f"{time}ms")

Filters:
==================================================

Tool Filter:
```pythontool = st.selectbox("Filter by tool",
["All", "Sentiment", "Summary", ...])if tool != "All":
history = [e for e in history if e['tool'] == tool.lower()]

Date Filter:
```pythonstart_date = st.date_input("Start date")
end_date = st.date_input("End date")Filter history by date range

Search:
```pythonsearch = st.text_input("Search in input")
if search:
history = [e for e in history
if search.lower() in e['input'].lower()]

Pagination:
==================================================

Pattern:
```pythonitems_per_page = 10
page = st.number_input("Page", 1, max_pages)start = (page - 1) * items_per_page
end = start + items_per_pagedisplay_items = history[start:end]

Or use session state for smoother UX.

Actions:
==================================================

Re-run Query:
```pythonif st.button("Re-run", key=f"rerun_{entry_id}"):
# Copy input to main interface
st.session_state.rerun_input = entry['input']
st.session_state.rerun_tool = entry['tool']

Delete Entry:
```pythonif st.button("Delete", key=f"delete_{entry_id}"):
history_manager.delete_entry(username, entry_id)
st.rerun()

Export:
```pythonif st.button("Export to CSV"):
df = pd.DataFrame(history)
csv = df.to_csv(index=False)
st.download_button("Download", csv, "history.csv")

Formatting:
==================================================

Timestamps:
```pythonfrom datetime import datetimetimestamp = datetime.fromisoformat(entry['timestamp'])
formatted = timestamp.strftime("%Y-%m-%d %H:%M:%S")

Truncation:
```pythondef truncate(text, length=50):
if len(text) > length:
return text[:length] + "..."
return text

Status Icons:
```pythonstatus = "✅" if entry['success'] else "❌"
"""

print("\n⏱️  Implementing history UI components...")

# ==================================================
# History Display Functions
# ==================================================

def show_recent_history(history_manager, username, limit=5):
    """
    Display recent history in sidebar.
    
    Args:
        history_manager: HistoryManager instance
        username: Username
        limit: Number of entries to show
    """
    st.sidebar.markdown("### 📜 Recent History")
    
    history = history_manager.get_history(username, limit=limit)
    
    if not history:
        st.sidebar.info("No history yet. Start using the tools!")
        return
    
    for entry in history:
        # Format timestamp
        ts = datetime.fromisoformat(entry['timestamp'])
        time_str = ts.strftime("%H:%M")
        
        # Truncate input
        input_preview = entry['input'][:30] + "..." if len(entry['input']) > 30 else entry['input']
        
        # Status icon
        status = "✅" if entry['success'] else "❌"
        
        # Tool emoji
        tool_emojis = {
            'sentiment': '😊',
            'summarizer': '📝',
            'fake_news': '🚨',
            'job_matcher': '💼'
        }
        emoji = tool_emojis.get(entry['tool'], '🔧')
        
        with st.sidebar.expander(f"{emoji} {time_str} - {entry['tool'].title()[:10]}"):
            st.caption(f"Status: {status}")
            st.text_area("Input", entry['input'], height=60, disabled=True, key=f"recent_{entry['id']}")
            
            if st.button("Re-run", key=f"rerun_recent_{entry['id']}"):
                st.session_state.rerun_input = entry['input']
                st.session_state.rerun_tool = entry['tool']
    
    if st.sidebar.button("📋 View Full History"):
        st.session_state.show_full_history = True

print("✅ show_recent_history() implemented")

# ==================================================
# Full History Page
# ==================================================

def show_full_history(history_manager, username):
    """
    Display full history with filters.
    
    Args:
        history_manager: HistoryManager instance
        username: Username
    """
    st.markdown("## 📜 Usage History")
    
    # Get all history
    history = history_manager.get_history(username)
    
    if not history:
        st.info("No history yet. Start using the tools to see your usage history here!")
        return
    
    # Filters
    col1, col2, col3 = st.columns([2, 2, 1])
    
    with col1:
        tool_filter = st.selectbox(
            "Filter by tool",
            ["All", "Sentiment", "Summarizer", "Fake News", "Job Matcher"]
        )
    
    with col2:
        search_query = st.text_input("Search in input", placeholder="Type to search...")
    
    with col3:
        st.write("")  # Spacing
        if st.button("🗑️ Clear History"):
            if st.checkbox("Confirm delete all"):
                history_manager.clear_history(username)
                st.success("History cleared!")
                st.rerun()
    
    # Apply filters
    filtered = history
    
    if tool_filter != "All":
        tool_key = tool_filter.lower().replace(" ", "_")
        filtered = [e for e in filtered if e['tool'] == tool_key]
    
    if search_query:
        filtered = [e for e in filtered 
                   if search_query.lower() in e['input'].lower()]
    
    # Stats
    st.markdown(f"**Showing {len(filtered)} of {len(history)} entries**")
    
    # Export
    if st.button("📥 Export to CSV"):
        df = pd.DataFrame(filtered)
        csv = df.to_csv(index=False)
        st.download_button(
            "Download CSV",
            csv,
            f"history_{username}_{datetime.now().strftime('%Y%m%d')}.csv",
            "text/csv"
        )
    
    st.markdown("---")
    
    # Display entries
    for entry in filtered:
        show_history_entry(entry, history_manager, username)

print("✅ show_full_history() implemented")

# ==================================================
# Single Entry Display
# ==================================================

def show_history_entry(entry, history_manager, username):
    """
    Display single history entry.
    
    Args:
        entry: History entry dict
        history_manager: HistoryManager instance
        username: Username
    """
    # Format timestamp
    ts = datetime.fromisoformat(entry['timestamp'])
    time_str = ts.strftime("%Y-%m-%d %H:%M:%S")
    
    # Status
    status = "✅ Success" if entry['success'] else "❌ Failed"
    status_color = "green" if entry['success'] else "red"
    
    # Tool emoji
    tool_emojis = {
        'sentiment': '😊',
        'summarizer': '📝',
        'fake_news': '🚨',
        'job_matcher': '💼'
    }
    emoji = tool_emojis.get(entry['tool'], '🔧')
    
    with st.expander(f"{emoji} {time_str} - {entry['tool'].title()} - {status}"):
        col1, col2 = st.columns([3, 1])
        
        with col1:
            st.markdown(f"**Tool:** {entry['tool'].title()}")
            st.markdown(f"**Status:** :{status_color}[{status}]")
            st.markdown(f"**Processing Time:** {entry.get('processing_time_ms', 'N/A')} ms")
        
        with col2:
            if st.button("🔄 Re-run", key=f"rerun_{entry['id']}"):
                st.session_state.rerun_input = entry['input']
                st.session_state.rerun_tool = entry['tool']
                st.success("Input copied! Go back to main page.")
            
            if st.button("🗑️ Delete", key=f"delete_{entry['id']}"):
                history_manager.delete_entry(username, entry['id'])
                st.success("Entry deleted!")
                time.sleep(0.5)
                st.rerun()
        
        st.markdown("**Input:**")
        st.text_area("", entry['input'], height=100, disabled=True, key=f"input_{entry['id']}")
        
        if entry['success']:
            st.markdown("**Result:**")
            st.json(entry['result'])
        else:
            st.markdown("**Error:**")
            st.error(entry.get('error', 'Unknown error'))

print("✅ show_history_entry() implemented")

# ==================================================
# Summary
# ==================================================

print("\n📊 History UI Summary:")

print("\n   Functions Created:")
print("      • show_recent_history(manager, username, limit=5)")
print("        - Sidebar widget")
print("        - Shows 5 most recent")
print("        - Expandable entries")
print("        - Re-run buttons")
print("      • show_full_history(manager, username)")
print("        - Full page display")
print("        - Filters (tool, search)")
print("        - Export to CSV")
print("        - Clear history")
print("      • show_history_entry(entry, manager, username)")
print("        - Single entry detail")
print("        - Re-run and delete actions")
print("        - Full input/result display")

print("\n🎨 UI Features:")
print("   • Recent history in sidebar (quick access)")
print("   • Full history page (all entries)")
print("   • Filters: By tool, search in input")
print("   • Actions: Re-run, delete, clear all")
print("   • Export: Download as CSV")
print("   • Status indicators: ✅ ❌")
print("   • Tool emojis: 😊 📝 🚨 💼")
print("   • Timestamp formatting")
print("   • Expandable cards")

print("\n✅ Exercise 2.3 Complete!")
print("="*80)


EXERCISE 2.3: Creating History Display Interface

⏱️  Implementing history UI components...
✅ show_recent_history() implemented
✅ show_full_history() implemented
✅ show_history_entry() implemented

📊 History UI Summary:

   Functions Created:
      • show_recent_history(manager, username, limit=5)
        - Sidebar widget
        - Shows 5 most recent
        - Expandable entries
        - Re-run buttons
      • show_full_history(manager, username)
        - Full page display
        - Filters (tool, search)
        - Export to CSV
        - Clear history
      • show_history_entry(entry, manager, username)
        - Single entry detail
        - Re-run and delete actions
        - Full input/result display

🎨 UI Features:
   • Recent history in sidebar (quick access)
   • Full history page (all entries)
   • Filters: By tool, search in input
   • Actions: Re-run, delete, clear all
   • Export: Download as CSV
   • Status indicators: ✅ ❌
   • Tool emojis: 😊 📝 🚨 💼
   • Timestamp format

In [14]:
# ==================================================
# EXERCISE 2.4: BUILD ANALYTICS DASHBOARD
# ==================================================

print("\n" + "="*80)
print("EXERCISE 2.4: Creating Analytics Dashboard")
print("="*80)

"""
📖 THEORY: Analytics Dashboard Design

Dashboard Purpose:
==================================================

Show User:
- How much they've used the app
- Which tools they prefer
- Success rates
- Activity patterns

Analytics Types:
==================================================

1. Summary Metrics:
   - Total queries
   - Success rate
   - Average processing time
   - Most used tool

2. Distribution Charts:
   - Queries by tool (bar chart)
   - Success/failure (pie chart)
   - Tool preference (%)

3. Time Series:
   - Activity over time (line chart)
   - Queries per day/week

4. Detailed Tables:
   - Recent activity
   - Performance by tool

Streamlit Metrics:
==================================================

st.metric():
```python
st.metric(
    label="Total Queries",
    value=42,
    delta="+12 this week"
)
```

Layout:
```python
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total", 42)
col2.metric("Success Rate", "95%")
col3.metric("Avg Time", "150ms")
col4.metric("Favorite", "Sentiment")
```

Charts with Plotly:
==================================================

Bar Chart (Tool Usage):
```python
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(x=list(by_tool.keys()), y=list(by_tool.values()))
])
fig.update_layout(title="Queries by Tool")
st.plotly_chart(fig)
```

Pie Chart (Success Rate):
```python
fig = go.Figure(data=[
    go.Pie(labels=['Success', 'Failed'], values=[95, 5])
])
st.plotly_chart(fig)
```

Line Chart (Activity):
```python
fig = go.Figure(data=[
    go.Scatter(x=dates, y=counts, mode='lines+markers')
])
fig.update_layout(title="Activity Over Time")
st.plotly_chart(fig)
```

Data Aggregation:
==================================================

By Tool:
```python
from collections import Counter
by_tool = Counter([e['tool'] for e in history])
```

By Date:
```python
from datetime import datetime
by_date = {}
for entry in history:
    date = datetime.fromisoformat(entry['timestamp']).date()
    by_date[date] = by_date.get(date, 0) + 1
```

Success Rate:
```python
total = len(history)
successes = sum(1 for e in history if e['success'])
rate = (successes / total) * 100
```

Dashboard Layout:
==================================================

Structure:
```
┌─────────────────────────────────────┐
│  Analytics Dashboard                │
├─────────────────────────────────────┤
│  [Metric] [Metric] [Metric] [Metric]│
├─────────────────────────────────────┤
│  ┌──────────────┐ ┌───────────────┐│
│  │ Bar Chart    │ │ Pie Chart     ││
│  │ (Tool Usage) │ │ (Success Rate)││
│  └──────────────┘ └───────────────┘│
├─────────────────────────────────────┤
│  ┌──────────────────────────────────┐
│  │ Line Chart (Activity Over Time) ││
│  └──────────────────────────────────┘
├─────────────────────────────────────┤
│  Recent Activity Table              │
└─────────────────────────────────────┘
```
"""

print("\n⏱️  Implementing analytics dashboard...")

# ==================================================
# Analytics Dashboard
# ==================================================

def show_analytics_dashboard(history_manager, username):
    """
    Display analytics dashboard for user.
    
    Args:
        history_manager: HistoryManager instance
        username: Username
    """
    st.markdown("## 📊 Analytics Dashboard")
    
    # Get analytics
    analytics = history_manager.get_analytics(username)
    history = history_manager.get_history(username)
    
    if analytics['total_queries'] == 0:
        st.info("No data yet. Start using the tools to see analytics!")
        return
    
    # Summary Metrics
    st.markdown("### 📈 Summary")
    
    col1, col2, col3, col4 = st.columns(4)
    
    with col1:
        st.metric("Total Queries", analytics['total_queries'])
    
    with col2:
        st.metric("Success Rate", f"{analytics['success_rate']:.1f}%")
    
    with col3:
        st.metric("Avg Time", f"{analytics['avg_processing_time']:.0f}ms")
    
    with col4:
        # Most used tool
        by_tool = analytics['by_tool']
        if by_tool:
            most_used = max(by_tool.items(), key=lambda x: x[1])[0]
            st.metric("Favorite Tool", most_used.title())
        else:
            st.metric("Favorite Tool", "N/A")
    
    st.markdown("---")
    
    # Charts
    col1, col2 = st.columns(2)
    
    # Bar Chart: Queries by Tool
    with col1:
        st.markdown("### 🔧 Queries by Tool")
        
        by_tool = analytics['by_tool']
        if by_tool:
            import plotly.graph_objects as go
            
            fig = go.Figure(data=[
                go.Bar(
                    x=[tool.title() for tool in by_tool.keys()],
                    y=list(by_tool.values()),
                    marker_color='#4CAF50'
                )
            ])
            fig.update_layout(
                xaxis_title="Tool",
                yaxis_title="Count",
                height=300
            )
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No tool usage data")
    
    # Pie Chart: Success Rate
    with col2:
        st.markdown("### ✅ Success vs Failed")
        
        successes = sum(1 for e in history if e['success'])
        failures = len(history) - successes
        
        if successes > 0 or failures > 0:
            import plotly.graph_objects as go
            
            fig = go.Figure(data=[
                go.Pie(
                    labels=['Success', 'Failed'],
                    values=[successes, failures],
                    marker_colors=['#4CAF50', '#f44336']
                )
            ])
            fig.update_layout(height=300)
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No success/failure data")
    
    st.markdown("---")
    
    # Activity Over Time
    st.markdown("### 📅 Activity Over Time")
    
    if len(history) > 1:
        # Group by date
        from collections import defaultdict
        by_date = defaultdict(int)
        
        for entry in history:
            date = datetime.fromisoformat(entry['timestamp']).date()
            by_date[date] += 1
        
        # Sort by date
        dates = sorted(by_date.keys())
        counts = [by_date[d] for d in dates]
        
        import plotly.graph_objects as go
        
        fig = go.Figure(data=[
            go.Scatter(
                x=dates,
                y=counts,
                mode='lines+markers',
                line=dict(color='#4CAF50', width=2),
                marker=dict(size=8)
            )
        ])
        fig.update_layout(
            xaxis_title="Date",
            yaxis_title="Queries",
            height=300
        )
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Need more data for timeline (minimum 2 queries)")
    
    st.markdown("---")
    
    # Performance Table
    st.markdown("### ⚡ Performance by Tool")
    
    # Calculate avg time per tool
    tool_performance = {}
    for entry in history:
        tool = entry['tool']
        time_ms = entry.get('processing_time_ms', 0)
        
        if tool not in tool_performance:
            tool_performance[tool] = []
        tool_performance[tool].append(time_ms)
    
    perf_data = []
    for tool, times in tool_performance.items():
        perf_data.append({
            'Tool': tool.title(),
            'Queries': len(times),
            'Avg Time (ms)': f"{sum(times)/len(times):.0f}",
            'Min Time (ms)': min(times),
            'Max Time (ms)': max(times)
        })
    
    if perf_data:
        df = pd.DataFrame(perf_data)
        st.dataframe(df, use_container_width=True)
    else:
        st.info("No performance data")

print("✅ show_analytics_dashboard() implemented")

print("\n📊 Analytics Dashboard Features:")

print("\n   Summary Metrics:")
print("      • Total Queries")
print("      • Success Rate (%)")
print("      • Average Processing Time (ms)")
print("      • Favorite Tool")

print("\n   Visualizations:")
print("      • Bar Chart: Queries by tool")
print("      • Pie Chart: Success vs failed")
print("      • Line Chart: Activity over time")
print("      • Table: Performance by tool")

print("\n   Data Aggregations:")
print("      • Count by tool")
print("      • Count by date")
print("      • Success/failure split")
print("      • Performance statistics")

print("\n🎨 Design:")
print("   • 4-column metric cards")
print("   • 2-column charts (bar + pie)")
print("   • Full-width timeline")
print("   • Performance table")
print("   • Green color theme (#4CAF50)")

print("\n✅ Exercise 2.4 Complete!")
print("="*80)


EXERCISE 2.4: Creating Analytics Dashboard

⏱️  Implementing analytics dashboard...
✅ show_analytics_dashboard() implemented

📊 Analytics Dashboard Features:

   Summary Metrics:
      • Total Queries
      • Success Rate (%)
      • Average Processing Time (ms)
      • Favorite Tool

   Visualizations:
      • Bar Chart: Queries by tool
      • Pie Chart: Success vs failed
      • Line Chart: Activity over time
      • Table: Performance by tool

   Data Aggregations:
      • Count by tool
      • Count by date
      • Success/failure split
      • Performance statistics

🎨 Design:
   • 4-column metric cards
   • 2-column charts (bar + pie)
   • Full-width timeline
   • Performance table
   • Green color theme (#4CAF50)

✅ Exercise 2.4 Complete!


In [15]:
print("\n" + "="*80)
print("🔑 PART 3: API KEY MANAGEMENT & RATE LIMITING")
print("="*80)


🔑 PART 3: API KEY MANAGEMENT & RATE LIMITING


In [16]:
# ==================================================
# EXERCISE 3.1: DESIGN API KEY SYSTEM
# ==================================================

print("\n" + "="*80)
print("EXERCISE 3.1: Planning API Key Architecture")
print("="*80)

"""
📖 THEORY: API Key Systems

What are API Keys?
==================================================

Purpose:
- Authentication for programmatic access
- Alternative to username/password
- Revocable (can disable without password change)
- Rate-limitable per key
- Trackable usage

Use Cases:
- Developers integrating tools
- Automated scripts
- Third-party applications
- CI/CD pipelines

API Key vs Password:
==================================================

Password:
- For human users
- Should be secret
- Changed occasionally
- Used in UI login

API Key:
- For programs/scripts
- Can be shared (carefully)
- Rotated regularly
- Used in API requests

API Key Format:
==================================================

Common Patterns:
1. Prefix + Random:
   sk_abc123def456...
   (sk = secret key)

2. UUID:
   123e4567-e89b-12d3-a456-426614174000

3. JWT Token:
   eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9...

Our Format: sk_{40 random chars}
- Prefix identifies type
- Random: 32 bytes → base64 → 40 chars
- Cryptographically secure

Generation:
```pythonimport secrets
import base64random_bytes = secrets.token_bytes(32)
key = base64.b64encode(random_bytes).decode()[:40]
api_key = f"sk_{key}"

API Key Storage:
==================================================

User Object:
```pythonuser = {
'username': 'alice',
'api_key': 'sk_abc123...',
'api_key_created': '2024-12-26T10:00:00',
'api_key_last_used': '2024-12-26T15:30:00',
'api_calls_count': 142
}

Separate API Keys File:
```json{
"sk_abc123...": {
"username": "alice",
"created": "2024-12-26T10:00:00",
"last_used": null,
"calls_count": 0,
"rate_limit": 100,
"active": true
}
}

Why Separate:
- Fast lookup (key → user)
- Can have multiple keys per user
- Easier to revoke

API Key Operations:
==================================================

1. Generate:
   - Create new key
   - Associate with user
   - Save to storage

2. Validate:
   - Check key exists
   - Check active status
   - Check not expired

3. Revoke:
   - Mark as inactive
   - Generate new key

4. Track Usage:
   - Increment call count
   - Update last_used
   - Check rate limits

5. Rotate:
   - Generate new key
   - Keep old valid for grace period
   - Deactivate old

API Key Best Practices:
==================================================

Security:
- Never log keys in plain text
- Mask in UI (show first/last 4 chars)
- Use HTTPS only
- Store securely (don't commit to git)

User Experience:
- One-click copy
- Show/hide toggle
- Regenerate button
- Clear warnings about security

Developer Experience:
- Clear documentation
- Code examples
- Rate limit info
- Error messages

API Endpoint Design:
==================================================

Authentication Header:
```pythonheaders = {
'Authorization': f'Bearer {api_key}'
}

Or query parameter:POST /api/sentiment?api_key=sk_abc123...

Validation Flow:
1. Extract key from request
2. Check key exists
3. Check active status
4. Check rate limit
5. Process request
6. Update usage stats

Response Codes:
- 200: Success
- 401: Invalid key
- 403: Inactive key
- 429: Rate limit exceeded

Example API Usage:
==================================================

Python:
```pythonimport requestsapi_key = "sk_abc123..."
url = "http://localhost:8501/api/sentiment"response = requests.post(
url,
headers={'Authorization': f'Bearer {api_key}'},
json={'text': 'This is great!'}
)result = response.json()

cURL:
```bashcurl -X POST \
-H "Authorization: Bearer sk_abc123..." \
-H "Content-Type: application/json" \
-d '{"text":"This is great!"}' \
http://localhost:8501/api/sentiment

Our Implementation:
==================================================

Scope: Web App Focus
- API keys generated
- UI shows key
- Can regenerate
- Track basic usage

Future: Full API
- Actual API endpoints
- Request validation
- Rate limiting
- Response formatting

Why Simple Now:
- Portfolio demo
- Shows concept
- Easy to expand
- Focus on ML tools
"""

print("\n⏱️  Designing API key system...")

print("\n🏗️  API Key Architecture:")

print("\n   Key Format:")
print("      • Prefix: 'sk_' (secret key)")
print("      • Length: 43 characters total (sk_ + 40 chars)")
print("      • Generation: secrets.token_bytes(32) → base64")
print("      • Example: sk_a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6q7r8")

print("\n   Storage Structure:")
print("      User Object (users.json):")
print("         • username")
print("         • api_key (current key)")
print("         • api_key_created")
print("         • api_key_last_used")
print("         • api_calls_count")

print("\n      Separate Keys File (api_keys.json):")
print("         • {api_key: {username, created, active, ...}}")
print("         • Fast lookup (key → user)")
print("         • Multiple keys per user (future)")

print("\n   Key Lifecycle:")
print("      1. Generate")
print("         • Create random key")
print("         • Store with user")
print("         • Save timestamp")
print("      2. Use")
print("         • Validate key exists")
print("         • Check active status")
print("         • Update last_used")
print("         • Increment counter")
print("      3. Regenerate")
print("         • Generate new key")
print("         • Replace old key")
print("         • Update user object")
print("      4. Revoke (future)")
print("         • Mark inactive")
print("         • Keep for audit")

print("\n🔒 Security Measures:")

print("\n   Key Generation:")
print("      • Cryptographically secure (secrets module)")
print("      • 32 bytes = 256 bits entropy")
print("      • Base64 encoding for safe transmission")
print("      • Unique per user")

print("\n   Key Display:")
print("      • Hidden by default (show dots)")
print("      • Show/hide toggle")
print("      • One-click copy")
print("      • Warning about security")

print("\n   Key Storage:")
print("      • Plain text in JSON (local files)")
print("      • Production: Hash keys (like passwords)")
print("      • Never in logs")
print("      • Not in git (.gitignore)")

print("\n📊 Usage Tracking:")

print("\n   Metrics per Key:")
print("      • Total API calls")
print("      • Last used timestamp")
print("      • Calls by tool")
print("      • Success/failure rate")

print("\n   Display:")
print("      • User dashboard")
print("      • API key section")
print("      • Usage statistics")
print("      • Regenerate button")

print("\n💡 User Interface:")

print("\n   API Key Section (Sidebar):")
code = '''
with st.sidebar.expander("🔑 API Key"):
    st.markdown("**Your API Key:**")
    
    # Show/hide toggle
    if show_key:
        st.code(user['api_key'])
        st.button("🙈 Hide")
    else:
        st.text("•" * 40)
        st.button("👁️ Show")
    
    # Copy button
    st.button("📋 Copy to Clipboard")
    
    # Regenerate
    if st.button("🔄 Regenerate Key"):
        # Confirm dialog
        # Generate new key
        # Update user
    
    # Usage stats
    st.metric("API Calls", user['api_calls_count'])
    st.caption(f"Created: {created_date}")
'''
print(code)

print("\n   API Documentation (Modal/Page):")
print("      • How to use API")
print("      • Authentication example")
print("      • Code samples (Python, cURL)")
print("      • Rate limits")
print("      • Error codes")

print("\n🎯 Implementation Components:")

print("\n   1. Key Generation:")
print("      • generate_api_key()")
print("      • Returns: 'sk_...' string")

print("\n   2. Key Validation:")
print("      • validate_api_key(key)")
print("      • Returns: user object or None")

print("\n   3. Key Regeneration:")
print("      • regenerate_api_key(username)")
print("      • Returns: new key")

print("\n   4. Usage Tracking:")
print("      • track_api_call(key, tool)")
print("      • Updates: last_used, call_count")

print("\n   5. UI Components:")
print("      • show_api_key_section(user)")
print("      • show_api_docs()")

print("\n✅ Exercise 3.1 Complete!")
print("="*80)


EXERCISE 3.1: Planning API Key Architecture

⏱️  Designing API key system...

🏗️  API Key Architecture:

   Key Format:
      • Prefix: 'sk_' (secret key)
      • Length: 43 characters total (sk_ + 40 chars)
      • Generation: secrets.token_bytes(32) → base64
      • Example: sk_a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6q7r8

   Storage Structure:
      User Object (users.json):
         • username
         • api_key (current key)
         • api_key_created
         • api_key_last_used
         • api_calls_count

      Separate Keys File (api_keys.json):
         • {api_key: {username, created, active, ...}}
         • Fast lookup (key → user)
         • Multiple keys per user (future)

   Key Lifecycle:
      1. Generate
         • Create random key
         • Store with user
         • Save timestamp
      2. Use
         • Validate key exists
         • Check active status
         • Update last_used
         • Increment counter
      3. Regenerate
         • Generate new key
         • Rep

In [17]:
# ==================================================
# EXERCISE 3.2: IMPLEMENT API KEY MANAGER
# ==================================================

print("\n" + "="*80)
print("EXERCISE 3.2: Building API Key Management System")
print("="*80)

"""
📖 THEORY: API Key Management Implementation

APIKeyManager Class:
==================================================

Responsibilities:
- Generate new keys
- Validate existing keys
- Track usage
- Regenerate keys
- Store/load key data

Methods:
- generate_key() → str
- validate_key(key) → user_data or None
- regenerate_key(username) → new_key
- track_usage(key, tool) → bool
- get_key_info(key) → dict

Storage Pattern:
==================================================

Two-way mapping:
1. User → Key (in users.json)
2. Key → User (in api_keys.json)

Why both:
- User login: Get key for user
- API request: Get user for key

File: api_keys.json
```json{
"sk_abc123...": {
"username": "alice",
"created": "2024-12-26T10:00:00",
"last_used": null,
"calls_count": 0,
"active": true
}
}

Thread Safety:
==================================================

Issue: Concurrent access
- Multiple requests at once
- File read/write conflicts

Solutions:
1. File locking (complex)
2. Database (overkill)
3. Accept risk (demo app)

Our approach: Simple (no locking)
- Demo/portfolio context
- Low concurrent usage
- Production: Use database

Validation Logic:
==================================================
```pythondef validate_key(key):
# Load keys
keys = load_keys()# Check exists
if key not in keys:
    return Nonekey_data = keys[key]# Check active
if not key_data.get('active', True):
    return None# Return user data
return key_data

Usage Tracking:
==================================================
```pythondef track_usage(key, tool):
keys = load_keys()if key in keys:
    # Update stats
    keys[key]['calls_count'] += 1
    keys[key]['last_used'] = datetime.now().isoformat()    # Track by tool
    if 'tools_used' not in keys[key]:
        keys[key]['tools_used'] = {}    keys[key]['tools_used'][tool] = \\
        keys[key]['tools_used'].get(tool, 0) + 1    save_keys(keys)
    return Truereturn False

Key Regeneration:
==================================================

Process:
1. Generate new key
2. Deactivate old key (optional: keep for grace period)
3. Update user object with new key
4. Add new key to api_keys.json
5. Return new key

Security consideration:
- Old key should be revoked immediately
- Or: grace period (24-48 hours) for transition
"""

print("\n⏱️  Implementing APIKeyManager class...")

# ==================================================
# APIKeyManager Class
# ==================================================

class APIKeyManager:
    """Manage API keys for users."""
    
    def __init__(self, keys_dir, user_manager):
        """
        Initialize APIKeyManager.
        
        Args:
            keys_dir: Path to API keys directory
            user_manager: UserManager instance
        """
        self.keys_dir = Path(keys_dir)
        self.keys_dir.mkdir(exist_ok=True)
        self.keys_file = self.keys_dir / "api_keys.json"
        self.user_manager = user_manager
    
    def load_keys(self):
        """Load API keys from file."""
        if self.keys_file.exists():
            try:
                with open(self.keys_file, 'r') as f:
                    return json.load(f)
            except:
                return {}
        return {}
    
    def save_keys(self, keys):
        """Save API keys to file."""
        try:
            with open(self.keys_file, 'w') as f:
                json.dump(keys, f, indent=2)
            return True
        except Exception as e:
            print(f"Error saving keys: {e}")
            return False
    
    def generate_key(self):
        """
        Generate new API key.
        
        Returns:
            str: API key in format 'sk_...'
        """
        random_bytes = secrets.token_bytes(32)
        key = base64.b64encode(random_bytes).decode()[:40]
        return f"sk_{key}"
    
    def create_key(self, username):
        """
        Create API key for user.
        
        Args:
            username: Username
        
        Returns:
            str: API key
        """
        # Generate key
        api_key = self.generate_key()
        
        # Create key data
        key_data = {
            'username': username,
            'created': datetime.now().isoformat(),
            'last_used': None,
            'calls_count': 0,
            'tools_used': {},
            'active': True
        }
        
        # Save to keys file
        keys = self.load_keys()
        keys[api_key] = key_data
        self.save_keys(keys)
        
        # Update user object
        self.user_manager.update_user(username, {
            'api_key': api_key,
            'api_key_created': datetime.now().isoformat()
        })
        
        return api_key
    
    def validate_key(self, api_key):
        """
        Validate API key.
        
        Args:
            api_key: API key string
        
        Returns:
            dict or None: Key data if valid, None if invalid
        """
        keys = self.load_keys()
        
        if api_key not in keys:
            return None
        
        key_data = keys[api_key]
        
        # Check active status
        if not key_data.get('active', True):
            return None
        
        return key_data
    
    def track_usage(self, api_key, tool):
        """
        Track API key usage.
        
        Args:
            api_key: API key string
            tool: Tool name
        
        Returns:
            bool: Success
        """
        keys = self.load_keys()
        
        if api_key not in keys:
            return False
        
        # Update stats
        keys[api_key]['calls_count'] += 1
        keys[api_key]['last_used'] = datetime.now().isoformat()
        
        # Track by tool
        if 'tools_used' not in keys[api_key]:
            keys[api_key]['tools_used'] = {}
        
        tool_key = tool.lower()
        keys[api_key]['tools_used'][tool_key] = \
            keys[api_key]['tools_used'].get(tool_key, 0) + 1
        
        # Save
        self.save_keys(keys)
        
        # Also update user object
        key_data = keys[api_key]
        username = key_data['username']
        user = self.user_manager.get_user(username)
        
        if user:
            self.user_manager.update_user(username, {
                'api_calls_count': keys[api_key]['calls_count']
            })
        
        return True
    
    def regenerate_key(self, username):
        """
        Regenerate API key for user.
        
        Args:
            username: Username
        
        Returns:
            str: New API key
        """
        # Get current key
        user = self.user_manager.get_user(username)
        old_key = user.get('api_key') if user else None
        
        # Deactivate old key
        if old_key:
            keys = self.load_keys()
            if old_key in keys:
                keys[old_key]['active'] = False
                self.save_keys(keys)
        
        # Create new key
        new_key = self.create_key(username)
        
        return new_key
    
    def get_key_info(self, api_key):
        """
        Get information about API key.
        
        Args:
            api_key: API key string
        
        Returns:
            dict or None: Key info
        """
        keys = self.load_keys()
        return keys.get(api_key)
    
    def revoke_key(self, api_key):
        """
        Revoke (deactivate) API key.
        
        Args:
            api_key: API key string
        
        Returns:
            bool: Success
        """
        keys = self.load_keys()
        
        if api_key in keys:
            keys[api_key]['active'] = False
            return self.save_keys(keys)
        
        return False

print("✅ APIKeyManager class implemented")

# ==================================================
# Test APIKeyManager
# ==================================================

print("\n📊 APIKeyManager Features:")

print("\n   Core Methods:")
print("      • generate_key()")
print("        Returns: 'sk_...' string")
print("      • create_key(username)")
print("        Returns: New API key, saves to storage")
print("      • validate_key(api_key)")
print("        Returns: Key data or None")
print("      • track_usage(api_key, tool)")
print("        Updates: calls_count, last_used, tools_used")
print("      • regenerate_key(username)")
print("        Deactivates old, creates new")
print("      • get_key_info(api_key)")
print("        Returns: Key metadata")
print("      • revoke_key(api_key)")
print("        Sets active=False")

print("\n💾 Storage:")
print("   • File: user_data/api_keys/api_keys.json")
print("   • Format: {api_key: key_data}")
print("   • Key data: username, created, last_used, calls_count, active")
print("   • Also updates user object in users.json")

print("\n🔒 Security:")
print("   • Keys stored as plain text (local demo)")
print("   • Production: Hash keys like passwords")
print("   • Active/inactive status")
print("   • Revocation support")

print("\n📈 Usage Tracking:")
print("   • Total calls per key")
print("   • Last used timestamp")
print("   • Calls per tool")
print("   • Active status")

print("\n✅ Exercise 3.2 Complete!")
print("="*80)


EXERCISE 3.2: Building API Key Management System

⏱️  Implementing APIKeyManager class...
✅ APIKeyManager class implemented

📊 APIKeyManager Features:

   Core Methods:
      • generate_key()
        Returns: 'sk_...' string
      • create_key(username)
        Returns: New API key, saves to storage
      • validate_key(api_key)
        Returns: Key data or None
      • track_usage(api_key, tool)
        Updates: calls_count, last_used, tools_used
      • regenerate_key(username)
        Deactivates old, creates new
      • get_key_info(api_key)
        Returns: Key metadata
      • revoke_key(api_key)
        Sets active=False

💾 Storage:
   • File: user_data/api_keys/api_keys.json
   • Format: {api_key: key_data}
   • Key data: username, created, last_used, calls_count, active
   • Also updates user object in users.json

🔒 Security:
   • Keys stored as plain text (local demo)
   • Production: Hash keys like passwords
   • Active/inactive status
   • Revocation support

📈 Usage Track

In [18]:
# ==================================================
# EXERCISE 3.3: IMPLEMENT RATE LIMITING
# ==================================================

print("\n" + "="*80)
print("EXERCISE 3.3: Building Rate Limiting System")
print("="*80)

"""
📖 THEORY: Rate Limiting

What is Rate Limiting?
==================================================

Purpose:
- Prevent abuse
- Fair resource allocation
- Protect infrastructure
- Manage costs

Common Patterns:
==================================================

1. Fixed Window:
   - 100 requests per hour
   - Window resets every hour
   - Simple but has burst issue

2. Sliding Window:
   - 100 requests in last 60 minutes
   - Smoother than fixed
   - More complex

3. Token Bucket:
   - Bucket holds tokens
   - Request consumes token
   - Tokens refill over time
   - Allows bursts

4. Leaky Bucket:
   - Requests queue
   - Process at fixed rate
   - Smooths traffic

Our Approach: Simple Fixed Window
==================================================

Strategy:
- Track requests per hour per user/key
- Reset counter every hour
- Simple to implement
- Good enough for demo

Implementation:
```pythonrate_limits = {
'alice': {
'count': 42,
'window_start': '2024-12-26T15:00:00',
'limit': 100
}
}def check_rate_limit(username):
# Get current window
current_hour = datetime.now().replace(
minute=0, second=0, microsecond=0
)# Load limits
limits = load_limits()if username not in limits:
    # First request
    limits[username] = {
        'count': 1,
        'window_start': current_hour.isoformat(),
        'limit': 100
    }
    return Trueuser_limit = limits[username]
window_start = datetime.fromisoformat(
    user_limit['window_start']
)# Check if new window
if current_hour > window_start:
    # Reset counter
    user_limit['count'] = 1
    user_limit['window_start'] = current_hour.isoformat()
    return True# Check limit
if user_limit['count'] >= user_limit['limit']:
    return False  # Rate limited!# Increment and allow
user_limit['count'] += 1
save_limits(limits)
return True

Rate Limit Tiers:
==================================================

Free Tier:
- 100 requests/hour
- All tools
- Basic support

Pro Tier (hypothetical):
- 1000 requests/hour
- Priority processing
- Premium support

Custom Headers:
==================================================

HTTP Response Headers:X-RateLimit-Limit: 100
X-RateLimit-Remaining: 42
X-RateLimit-Reset: 1640530800

User feedback on limits.

Error Response:
==================================================

429 Too Many Requests:
```json{
"error": "Rate limit exceeded",
"limit": 100,
"window": "hour",
"reset_at": "2024-12-26T16:00:00"
}

UI Display:
==================================================

Show user their limits:
- Current usage: 42/100
- Reset time: "Resets in 23 minutes"
- Progress bar
- Warning at 80%

Rate Limit Configuration:
==================================================

Per User Type:
```pythonRATE_LIMITS = {
'guest': 10,      # 10/hour
'free': 100,      # 100/hour
'pro': 1000,      # 1000/hour
'unlimited': None # No limit
}

Configurable per tool (future):
- Sentiment: Fast, high limit
- Summarizer: Slow, lower limit
- Batch: Much lower limit

Why Rate Limiting Matters:
==================================================

Protection:
- Prevent single user monopolizing
- Protect against attacks
- Manage server load

Business:
- Monetization path
- Tiered pricing
- Fair usage

User Experience:
- Clear expectations
- Visible limits
- Predictable service
"""

print("\n⏱️  Implementing rate limiting system...")

# ==================================================
# RateLimiter Class
# ==================================================

class RateLimiter:
    """Simple rate limiting system."""
    
    def __init__(self, limits_dir):
        """
        Initialize RateLimiter.
        
        Args:
            limits_dir: Path to rate limits directory
        """
        self.limits_dir = Path(limits_dir)
        self.limits_dir.mkdir(exist_ok=True)
        self.limits_file = self.limits_dir / "rate_limits.json"
        
        # Default limits per hour
        self.default_limits = {
            'guest': 10,
            'registered': 100,
            'pro': 1000
        }
    
    def load_limits(self):
        """Load rate limits from file."""
        if self.limits_file.exists():
            try:
                with open(self.limits_file, 'r') as f:
                    return json.load(f)
            except:
                return {}
        return {}
    
    def save_limits(self, limits):
        """Save rate limits to file."""
        try:
            with open(self.limits_file, 'w') as f:
                json.dump(limits, f, indent=2)
            return True
        except:
            return False
    
    def get_user_limit(self, username, is_guest=False):
        """
        Get rate limit for user.
        
        Args:
            username: Username
            is_guest: Is guest user
        
        Returns:
            int: Requests per hour allowed
        """
        if is_guest:
            return self.default_limits['guest']
        else:
            return self.default_limits['registered']
    
    def check_rate_limit(self, username, is_guest=False):
        """
        Check if user has exceeded rate limit.
        
        Args:
            username: Username
            is_guest: Is guest user
        
        Returns:
            tuple: (allowed: bool, remaining: int, reset_time: str)
        """
        # Get current hour window
        current_hour = datetime.now().replace(
            minute=0, second=0, microsecond=0
        )
        
        # Load limits
        limits = self.load_limits()
        
        # Get user's limit
        user_limit_value = self.get_user_limit(username, is_guest)
        
        # Initialize if first request
        if username not in limits:
            limits[username] = {
                'count': 1,
                'window_start': current_hour.isoformat(),
                'limit': user_limit_value
            }
            self.save_limits(limits)
            
            reset_time = (current_hour + timedelta(hours=1)).isoformat()
            return True, user_limit_value - 1, reset_time
        
        user_data = limits[username]
        window_start = datetime.fromisoformat(user_data['window_start'])
        
        # Check if new window (hour passed)
        if current_hour > window_start:
            # Reset counter for new window
            user_data['count'] = 1
            user_data['window_start'] = current_hour.isoformat()
            user_data['limit'] = user_limit_value
            self.save_limits(limits)
            
            reset_time = (current_hour + timedelta(hours=1)).isoformat()
            return True, user_limit_value - 1, reset_time
        
        # Check if limit exceeded
        if user_data['count'] >= user_data['limit']:
            reset_time = (window_start + timedelta(hours=1)).isoformat()
            return False, 0, reset_time
        
        # Increment and allow
        user_data['count'] += 1
        self.save_limits(limits)
        
        remaining = user_data['limit'] - user_data['count']
        reset_time = (window_start + timedelta(hours=1)).isoformat()
        
        return True, remaining, reset_time
    
    def get_usage_stats(self, username):
        """
        Get current usage stats for user.
        
        Args:
            username: Username
        
        Returns:
            dict: {count, limit, remaining, reset_time}
        """
        limits = self.load_limits()
        
        if username not in limits:
            return {
                'count': 0,
                'limit': self.get_user_limit(username),
                'remaining': self.get_user_limit(username),
                'reset_time': None
            }
        
        user_data = limits[username]
        window_start = datetime.fromisoformat(user_data['window_start'])
        reset_time = (window_start + timedelta(hours=1)).isoformat()
        
        return {
            'count': user_data['count'],
            'limit': user_data['limit'],
            'remaining': user_data['limit'] - user_data['count'],
            'reset_time': reset_time
        }
    
    def reset_user_limit(self, username):
        """
        Manually reset user's rate limit.
        
        Args:
            username: Username
        
        Returns:
            bool: Success
        """
        limits = self.load_limits()
        
        if username in limits:
            del limits[username]
            return self.save_limits(limits)
        
        return True

print("✅ RateLimiter class implemented")

# ==================================================
# Usage Display Function
# ==================================================

def show_rate_limit_status(rate_limiter, username, is_guest=False):
    """
    Display rate limit status for user.
    
    Args:
        rate_limiter: RateLimiter instance
        username: Username
        is_guest: Is guest user
    """
    stats = rate_limiter.get_usage_stats(username)
    
    st.markdown("### ⏱️ Rate Limit Status")
    
    # Progress bar
    progress = stats['count'] / stats['limit'] if stats['limit'] > 0 else 0
    st.progress(progress)
    
    # Metrics
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric("Used", stats['count'])
    
    with col2:
        st.metric("Remaining", stats['remaining'])
    
    with col3:
        st.metric("Limit", stats['limit'])
    
    # Reset time
    if stats['reset_time']:
        reset_dt = datetime.fromisoformat(stats['reset_time'])
        now = datetime.now()
        
        if reset_dt > now:
            delta = reset_dt - now
            minutes = delta.seconds // 60
            st.caption(f"Resets in {minutes} minutes")
        else:
            st.caption("Resets next hour")
    
    # Warning if close to limit
    if stats['remaining'] <= stats['limit'] * 0.2:  # 20% or less remaining
        st.warning(f"⚠️ You're approaching your rate limit ({stats['remaining']} requests remaining)")

print("✅ show_rate_limit_status() implemented")

print("\n📊 Rate Limiting Features:")

print("\n   RateLimiter Class:")
print("      • check_rate_limit(username, is_guest)")
print("        Returns: (allowed, remaining, reset_time)")
print("      • get_usage_stats(username)")
print("        Returns: {count, limit, remaining, reset_time}")
print("      • reset_user_limit(username)")
print("        Manually reset (admin function)")
print("      • get_user_limit(username, is_guest)")
print("        Returns: Limit value for user tier")

print("\n⚙️ Rate Limit Configuration:")
print("   • Guest users: 10 requests/hour")
print("   • Registered users: 100 requests/hour")
print("   • Pro users (future): 1000 requests/hour")
print("   • Fixed window: Resets every hour")

print("\n📊 UI Components:")
print("   • Progress bar (visual usage)")
print("   • Metrics: Used, Remaining, Limit")
print("   • Reset time countdown")
print("   • Warning at 80% usage")

print("\n🔒 Protection:")
print("   • Prevents abuse")
print("   • Fair resource allocation")
print("   • Per-user tracking")
print("   • Automatic window reset")

print("\n✅ Exercise 3.3 Complete!")
print("="*80)


EXERCISE 3.3: Building Rate Limiting System

⏱️  Implementing rate limiting system...
✅ RateLimiter class implemented
✅ show_rate_limit_status() implemented

📊 Rate Limiting Features:

   RateLimiter Class:
      • check_rate_limit(username, is_guest)
        Returns: (allowed, remaining, reset_time)
      • get_usage_stats(username)
        Returns: {count, limit, remaining, reset_time}
      • reset_user_limit(username)
        Manually reset (admin function)
      • get_user_limit(username, is_guest)
        Returns: Limit value for user tier

⚙️ Rate Limit Configuration:
   • Guest users: 10 requests/hour
   • Registered users: 100 requests/hour
   • Pro users (future): 1000 requests/hour
   • Fixed window: Resets every hour

📊 UI Components:
   • Progress bar (visual usage)
   • Metrics: Used, Remaining, Limit
   • Reset time countdown
   • Warning at 80% usage

🔒 Protection:
   • Prevents abuse
   • Fair resource allocation
   • Per-user tracking
   • Automatic window reset

✅ E

In [19]:
# ==================================================
# EXERCISE 3.4: BUILD API DOCUMENTATION
# ==================================================

print("\n" + "="*80)
print("EXERCISE 3.4: Creating API Documentation")
print("="*80)

"""
📖 THEORY: API Documentation

Why Documentation Matters:
==================================================

Developers need to know:
- How to authenticate
- What endpoints exist
- Request/response format
- Error codes
- Rate limits
- Code examples

Good Documentation:
- Clear and concise
- Code examples
- Try-it-yourself
- Common errors
- Quick start guide

Documentation Sections:
==================================================

1. Quick Start:
   - Get API key
   - First API call
   - See results

2. Authentication:
   - Header format
   - Example request

3. Endpoints:
   - URL
   - Method (POST/GET)
   - Parameters
   - Response format

4. Rate Limits:
   - Limits by tier
   - Headers
   - Error handling

5. Code Examples:
   - Python
   - JavaScript
   - cURL
   - Multiple languages

6. Error Codes:
   - 200: Success
   - 400: Bad request
   - 401: Invalid key
   - 429: Rate limited
   - 500: Server error

Example Documentation:
==================================================

Quick Start:
````markdown
## Quick Start

1. Get your API key from the dashboard
2. Make your first request:
```python
import requests

api_key = "sk_your_key_here"
url = "http://localhost:8501/api/sentiment"

response = requests.post(
    url,
    headers={'Authorization': f'Bearer {api_key}'},
    json={'text': 'This is great!'}
)

print(response.json())
```

3. See the result!
````

Authentication Section:
````markdown
## Authentication

Include your API key in the Authorization header:
````
Authorization: Bearer sk_your_key_here
Example:
pythonheaders = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json'
}
````
````

Endpoint Documentation:
````markdown
## POST /api/sentiment

Analyze sentiment of text.

**Request:**
```json
{
  "text": "Your text here"
}
```

**Response:**
```json
{
  "sentiment": "Positive",
  "confidence": 95.2,
  "scores": {
    "negative": 4.8,
    "positive": 95.2
  }
}
```

**Rate Limit:** 100 requests/hour
````

Interactive Documentation:
==================================================

Swagger/OpenAPI:
- Auto-generated
- Try-it-yourself
- Interactive
- Standard format

Streamlit Implementation:
- Markdown pages
- Code blocks
- Copy buttons
- Live examples

Our Implementation:
==================================================

Simple but Complete:
- Markdown documentation
- Code examples (copyable)
- Clear structure
- Easy to maintain

Future Enhancements:
- Actual API endpoints
- OpenAPI spec
- Postman collection
- SDKs (Python, JS)
"""

print("\n⏱️  Creating API documentation...")

# ==================================================
# API Documentation Function
# ==================================================

def show_api_documentation():
    """Display API documentation page."""
    
    st.markdown("# 📖 API Documentation")
    st.markdown("Learn how to integrate TextAI Studio into your applications.")
    
    # Quick Start
    st.markdown("## 🚀 Quick Start")
    
    st.markdown("""
    1. **Get your API key** from the sidebar (🔑 API Key section)
    2. **Install requests library** (if needed): `pip install requests`
    3. **Make your first request:**
    """)
    
    quick_start_code = '''import requests

# Your API key from the dashboard
api_key = "sk_your_key_here"

# API endpoint (when deployed)
url = "https://your-app.streamlit.app/api/sentiment"

# Make request
response = requests.post(
    url,
    headers={'Authorization': f'Bearer {api_key}'},
    json={'text': 'This product is amazing!'}
)

# Get result
result = response.json()
print(result)
'''
    
    st.code(quick_start_code, language='python')
    
    if st.button("📋 Copy Quick Start Code"):
        st.success("Code copied! (In actual app, would copy to clipboard)")
    
    st.markdown("---")
    
    # Authentication
    st.markdown("## 🔐 Authentication")
    
    st.markdown("""
    All API requests require authentication using your API key.
    
    Include your API key in the `Authorization` header:
    """)
    
    auth_code = '''Authorization: Bearer sk_your_key_here'''
    st.code(auth_code, language='text')
    
    st.markdown("**Python example:**")
    auth_python = '''headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json'
}'''
    st.code(auth_python, language='python')
    
    st.markdown("---")
    
    # Endpoints
    st.markdown("## 🔧 API Endpoints")
    
    # Sentiment Analysis
    with st.expander("POST /api/sentiment - Sentiment Analysis"):
        st.markdown("Analyze the emotional tone of text (Positive/Negative).")
        
        st.markdown("**Request:**")
        st.code('''POST /api/sentiment
Content-Type: application/json
Authorization: Bearer sk_your_key

{
  "text": "Your text to analyze"
}''', language='json')
        
        st.markdown("**Response:**")
        st.code('''{
  "sentiment": "Positive",
  "confidence": 95.2,
  "scores": {
    "negative": 4.8,
    "positive": 95.2
  },
  "processing_time_ms": 127
}''', language='json')
        
        st.markdown("**Example (Python):**")
        st.code('''response = requests.post(
    "https://your-app.streamlit.app/api/sentiment",
    headers={'Authorization': f'Bearer {api_key}'},
    json={'text': 'This is amazing!'}
)''', language='python')
    
    # Text Summarization
    with st.expander("POST /api/summarize - Text Summarization"):
        st.markdown("Generate concise summary of long text.")
        
        st.markdown("**Request:**")
        st.code('''POST /api/summarize
Content-Type: application/json
Authorization: Bearer sk_your_key

{
  "text": "Long article text...",
  "length": "medium"
}''', language='json')
        
        st.markdown("**Parameters:**")
        st.markdown("- `length`: 'short', 'medium', or 'long'")
        
        st.markdown("**Response:**")
        st.code('''{
  "summary": "Generated summary text...",
  "original_words": 500,
  "summary_words": 125,
  "compression_ratio": 75,
  "processing_time_ms": 843
}''', language='json')
    
    # Fake News Detection
    with st.expander("POST /api/fake-news - Fake News Detection"):
        st.markdown("Check credibility of news articles.")
        
        st.markdown("**Request:**")
        st.code('''POST /api/fake-news
Content-Type: application/json
Authorization: Bearer sk_your_key

{
  "text": "News article text..."
}''', language='json')
        
        st.markdown("**Response:**")
        st.code('''{
  "prediction": "Real",
  "confidence": 87.3,
  "scores": {
    "real": 87.3,
    "fake": 12.7
  },
  "processing_time_ms": 156
}''', language='json')
    
    st.markdown("---")
    
    # Rate Limits
    st.markdown("## ⏱️ Rate Limits")
    
    st.markdown("""
    Rate limits apply per API key per hour:
    
    - **Guest**: 10 requests/hour
    - **Registered**: 100 requests/hour
    - **Pro**: 1000 requests/hour *(coming soon)*
    
    **Response Headers:**
    """)
    
    st.code('''X-RateLimit-Limit: 100
X-RateLimit-Remaining: 42
X-RateLimit-Reset: 2024-12-26T16:00:00''', language='text')
    
    st.markdown("**Rate Limit Exceeded (429):**")
    st.code('''{
  "error": "Rate limit exceeded",
  "limit": 100,
  "reset_at": "2024-12-26T16:00:00"
}''', language='json')
    
    st.markdown("---")
    
    # Error Codes
    st.markdown("## ❌ Error Codes")
    
    errors_df = pd.DataFrame({
        'Code': [200, 400, 401, 403, 429, 500],
        'Status': ['OK', 'Bad Request', 'Unauthorized', 'Forbidden', 'Too Many Requests', 'Server Error'],
        'Description': [
            'Success',
            'Invalid request format',
            'Invalid or missing API key',
            'Inactive API key',
            'Rate limit exceeded',
            'Internal server error'
        ]
    })
    
    st.dataframe(errors_df, use_container_width=True)
    
    st.markdown("---")
    
    # Code Examples
    st.markdown("## 💻 Code Examples")
    
    tab1, tab2, tab3 = st.tabs(["Python", "JavaScript", "cURL"])
    
    with tab1:
        st.markdown("**Python (requests):**")
        python_code = '''import requests

api_key = "sk_your_key_here"
base_url = "https://your-app.streamlit.app/api"

# Sentiment Analysis
response = requests.post(
    f"{base_url}/sentiment",
    headers={'Authorization': f'Bearer {api_key}'},
    json={'text': 'This is great!'}
)

print(response.json())

# Text Summarization
response = requests.post(
    f"{base_url}/summarize",
    headers={'Authorization': f'Bearer {api_key}'},
    json={
        'text': 'Long article...',
        'length': 'medium'
    }
)

print(response.json())'''
        st.code(python_code, language='python')
    
    with tab2:
        st.markdown("**JavaScript (fetch):**")
        js_code = '''const apiKey = "sk_your_key_here";
const baseUrl = "https://your-app.streamlit.app/api";

// Sentiment Analysis
fetch(`${baseUrl}/sentiment`, {
  method: 'POST',
  headers: {
    'Authorization': `Bearer ${apiKey}`,
    'Content-Type': 'application/json'
  },
  body: JSON.stringify({
    text: 'This is great!'
  })
})
.then(res => res.json())
.then(data => console.log(data));'''
        st.code(js_code, language='javascript')
    
    with tab3:
        st.markdown("**cURL:**")
        curl_code = '''# Sentiment Analysis
curl -X POST \\
  -H "Authorization: Bearer sk_your_key_here" \\
  -H "Content-Type: application/json" \\
  -d '{"text":"This is great!"}' \\
  https://your-app.streamlit.app/api/sentiment

# Text Summarization
curl -X POST \\
  -H "Authorization: Bearer sk_your_key_here" \\
  -H "Content-Type: application/json" \\
  -d '{"text":"Long article...","length":"medium"}' \\
  https://your-app.streamlit.app/api/summarize'''
        st.code(curl_code, language='bash')
    
    st.markdown("---")
    
    # Support
    st.markdown("## 💬 Support")
    
    st.info("""
    **Need help?**
    
    - Check the examples above
    - Review error codes
    - Test with small requests first
    - Contact: support@textai.studio *(example)*
    """)

print("✅ show_api_documentation() implemented")

print("\n📖 API Documentation Features:")

print("\n   Sections:")
print("      • Quick Start (get started fast)")
print("      • Authentication (how to use API keys)")
print("      • Endpoints (all available APIs)")
print("      • Rate Limits (usage limits)")
print("      • Error Codes (troubleshooting)")
print("      • Code Examples (Python, JS, cURL)")
print("      • Support (contact info)")

print("\n🎨 Interactive Features:")
print("   • Expandable endpoint docs")
print("   • Tabbed code examples")
print("   • Copyable code blocks")
print("   • Request/response samples")
print("   • Error code table")

print("\n📝 Developer Experience:")
print("   • Clear structure")
print("   • Complete examples")
print("   • Multiple languages")
print("   • Copy-paste ready")
print("   • Production-quality docs")

print("\n✅ Exercise 3.4 Complete!")
print("="*80)


EXERCISE 3.4: Creating API Documentation

⏱️  Creating API documentation...
✅ show_api_documentation() implemented

📖 API Documentation Features:

   Sections:
      • Quick Start (get started fast)
      • Authentication (how to use API keys)
      • Endpoints (all available APIs)
      • Rate Limits (usage limits)
      • Error Codes (troubleshooting)
      • Code Examples (Python, JS, cURL)
      • Support (contact info)

🎨 Interactive Features:
   • Expandable endpoint docs
   • Tabbed code examples
   • Copyable code blocks
   • Request/response samples
   • Error code table

📝 Developer Experience:
   • Clear structure
   • Complete examples
   • Multiple languages
   • Copy-paste ready
   • Production-quality docs

✅ Exercise 3.4 Complete!


In [20]:
print("\n" + "="*80)
print("🎯 PART 4: TESTING, DOCUMENTATION & SUMMARY")
print("="*80)


🎯 PART 4: TESTING, DOCUMENTATION & SUMMARY


In [21]:
# ==================================================
# EXERCISE 4.1: COMPREHENSIVE TESTING
# ==================================================

print("\n" + "="*80)
print("EXERCISE 4.1: Testing All Day 59 Components")
print("="*80)

"""
📖 THEORY: Testing User Management Systems

Testing Strategy:
==================================================

Day 59 Components:
1. User authentication
2. Usage history tracking
3. API key management
4. Rate limiting

Testing Levels:
- Unit: Individual functions
- Integration: Components together
- System: End-to-end flows
- Manual: UI testing

Test Categories:
==================================================

1. Authentication Tests:
   ✅ User registration
   ✅ Login validation
   ✅ Password hashing
   ✅ Session management
   ✅ Guest mode

2. History Tests:
   ✅ Entry creation
   ✅ History retrieval
   ✅ Filtering (tool, date)
   ✅ Analytics calculation
   ✅ Entry deletion

3. API Key Tests:
   ✅ Key generation
   ✅ Key validation
   ✅ Usage tracking
   ✅ Key regeneration
   ✅ Key revocation

4. Rate Limit Tests:
   ✅ Limit enforcement
   ✅ Window reset
   ✅ Usage tracking
   ✅ Different tiers

Edge Cases:
==================================================

- Concurrent access
- File corruption
- Disk full
- Invalid data
- Missing files
- Edge timestamps
"""

print("\n⏱️  Running comprehensive tests...")

print("\n🧪 Test Suite - Day 59 Components:")

# ==================================================
# Test Suite 1: User Authentication
# ==================================================

print("\n   Test Suite 1: User Authentication")

try:
    # Initialize managers
    test_user_mgr = UserManager(USER_DATA_DIR)
    
    # Test 1.1: User registration
    print("\n      Test 1.1: User Registration")
    success, msg = test_user_mgr.register_user("test_day59_user", "testpass123", "test@example.com")
    if success:
        print(f"         ✅ Registration successful: {msg}")
    else:
        print(f"         ⚠️ Registration: {msg}")
    
    # Test 1.2: Duplicate registration
    print("\n      Test 1.2: Duplicate Registration")
    success, msg = test_user_mgr.register_user("test_day59_user", "newpass123", "new@example.com")
    if not success and "exists" in msg:
        print(f"         ✅ Duplicate correctly rejected: {msg}")
    else:
        print(f"         ❌ Duplicate not properly rejected: {msg}")
    
    # Test 1.3: Login
    print("\n      Test 1.3: User Login")
    user = test_user_mgr.login_user("test_day59_user", "testpass123")
    if user:
        print(f"         ✅ Login successful for: {user['username']}")
    else:
        print("         ❌ Login failed")
    
    # Test 1.4: Wrong password
    print("\n      Test 1.4: Wrong Password")
    user = test_user_mgr.login_user("test_day59_user", "wrongpass")
    if user is None:
        print("         ✅ Wrong password correctly rejected")
    else:
        print("         ❌ Wrong password incorrectly accepted")
    
    print("\n      ✅ Authentication Suite Complete")

except Exception as e:
    print(f"\n      ❌ Authentication Suite Failed: {e}")

# ==================================================
# Test Suite 2: Usage History
# ==================================================

print("\n   Test Suite 2: Usage History")

try:
    # Initialize history manager
    test_history_mgr = HistoryManager(HISTORY_DIR)
    
    # Test 2.1: Add entry
    print("\n      Test 2.1: Add History Entry")
    entry_id = test_history_mgr.add_entry(
        username="test_day59_user",
        tool="sentiment",
        input_text="Test input",
        result={"sentiment": "Positive", "confidence": 95.0},
        processing_time_ms=100,
        success=True
    )
    if entry_id:
        print(f"         ✅ Entry created: {entry_id}")
    else:
        print("         ❌ Entry creation failed")
    
    # Test 2.2: Retrieve history
    print("\n      Test 2.2: Retrieve History")
    history = test_history_mgr.get_history("test_day59_user")
    if len(history) >= 1:
        print(f"         ✅ History retrieved: {len(history)} entries")
    else:
        print("         ❌ History retrieval failed")
    
    # Test 2.3: Filter by tool
    print("\n      Test 2.3: Filter by Tool")
    sentiment_history = test_history_mgr.get_by_tool("test_day59_user", "sentiment")
    if len(sentiment_history) >= 1:
        print(f"         ✅ Filtered history: {len(sentiment_history)} sentiment entries")
    else:
        print("         ⚠️ No sentiment entries found")
    
    # Test 2.4: Analytics
    print("\n      Test 2.4: Calculate Analytics")
    analytics = test_history_mgr.get_analytics("test_day59_user")
    if analytics['total_queries'] >= 1:
        print(f"         ✅ Analytics calculated: {analytics['total_queries']} total queries")
        print(f"            Success rate: {analytics['success_rate']:.1f}%")
    else:
        print("         ❌ Analytics calculation failed")
    
    # Test 2.5: Delete entry
    print("\n      Test 2.5: Delete Entry")
    if test_history_mgr.delete_entry("test_day59_user", entry_id):
        print("         ✅ Entry deleted successfully")
    else:
        print("         ❌ Entry deletion failed")
    
    print("\n      ✅ History Suite Complete")

except Exception as e:
    print(f"\n      ❌ History Suite Failed: {e}")

# ==================================================
# Test Suite 3: API Key Management
# ==================================================

print("\n   Test Suite 3: API Key Management")

try:
    # Initialize API key manager
    test_api_mgr = APIKeyManager(API_KEYS_DIR, test_user_mgr)
    
    # Test 3.1: Generate key
    print("\n      Test 3.1: Generate API Key")
    key = test_api_mgr.generate_key()
    if key.startswith("sk_") and len(key) == 43:
        print(f"         ✅ Key generated: {key[:20]}...")
    else:
        print(f"         ❌ Key format incorrect: {key}")
    
    # Test 3.2: Create key for user
    print("\n      Test 3.2: Create Key for User")
    user_key = test_api_mgr.create_key("test_day59_user")
    if user_key:
        print(f"         ✅ User key created: {user_key[:20]}...")
    else:
        print("         ❌ User key creation failed")
    
    # Test 3.3: Validate key
    print("\n      Test 3.3: Validate API Key")
    key_data = test_api_mgr.validate_key(user_key)
    if key_data and key_data['username'] == "test_day59_user":
        print(f"         ✅ Key validated: {key_data['username']}")
    else:
        print("         ❌ Key validation failed")
    
    # Test 3.4: Track usage
    print("\n      Test 3.4: Track API Usage")
    if test_api_mgr.track_usage(user_key, "sentiment"):
        print("         ✅ Usage tracked")
        key_info = test_api_mgr.get_key_info(user_key)
        print(f"            Calls: {key_info['calls_count']}")
    else:
        print("         ❌ Usage tracking failed")
    
    # Test 3.5: Regenerate key
    print("\n      Test 3.5: Regenerate API Key")
    new_key = test_api_mgr.regenerate_key("test_day59_user")
    if new_key and new_key != user_key:
        print(f"         ✅ Key regenerated: {new_key[:20]}...")
        
        # Old key should be inactive
        old_key_data = test_api_mgr.validate_key(user_key)
        if old_key_data is None:
            print("         ✅ Old key deactivated")
        else:
            print("         ⚠️ Old key still active")
    else:
        print("         ❌ Key regeneration failed")
    
    print("\n      ✅ API Key Suite Complete")

except Exception as e:
    print(f"\n      ❌ API Key Suite Failed: {e}")

# ==================================================
# Test Suite 4: Rate Limiting
# ==================================================

print("\n   Test Suite 4: Rate Limiting")

try:
    # Initialize rate limiter
    test_limiter = RateLimiter(USER_DATA_DIR / "rate_limits")
    
    # Test 4.1: Check limit (first request)
    print("\n      Test 4.1: First Request")
    allowed, remaining, reset = test_limiter.check_rate_limit("test_rate_user", is_guest=False)
    if allowed:
        print(f"         ✅ Request allowed, remaining: {remaining}")
    else:
        print("         ❌ First request blocked")
    
    # Test 4.2: Multiple requests
    print("\n      Test 4.2: Multiple Requests")
    for i in range(5):
        allowed, remaining, reset = test_limiter.check_rate_limit("test_rate_user", is_guest=False)
    
    if allowed:
        print(f"         ✅ Multiple requests allowed, remaining: {remaining}")
    else:
        print("         ⚠️ Rate limited after 5 requests")
    
    # Test 4.3: Get usage stats
    print("\n      Test 4.3: Usage Statistics")
    stats = test_limiter.get_usage_stats("test_rate_user")
    if stats['count'] >= 5:
        print(f"         ✅ Stats tracked: {stats['count']}/{stats['limit']}")
    else:
        print(f"         ⚠️ Stats: {stats}")
    
    # Test 4.4: Guest vs Registered limits
    print("\n      Test 4.4: Guest vs Registered Limits")
    guest_limit = test_limiter.get_user_limit("guest_user", is_guest=True)
    reg_limit = test_limiter.get_user_limit("reg_user", is_guest=False)
    
    if guest_limit < reg_limit:
        print(f"         ✅ Guest limit ({guest_limit}) < Registered limit ({reg_limit})")
    else:
        print(f"         ❌ Limits incorrect: Guest={guest_limit}, Reg={reg_limit}")
    
    print("\n      ✅ Rate Limiting Suite Complete")

except Exception as e:
    print(f"\n      ❌ Rate Limiting Suite Failed: {e}")

# ==================================================
# Test Summary
# ==================================================

print("\n📊 Test Summary:")
print("   ✅ Authentication: Registration, login, password validation")
print("   ✅ History: Add, retrieve, filter, analytics, delete")
print("   ✅ API Keys: Generate, validate, track, regenerate")
print("   ✅ Rate Limiting: Enforcement, stats, tiers")

print("\n💡 Manual UI Tests (Run in Streamlit app):")
print("   1. Guest mode button")
print("   2. Login/Signup forms")
print("   3. User info display")
print("   4. History sidebar")
print("   5. Full history page")
print("   6. Analytics dashboard")
print("   7. API key show/hide")
print("   8. Rate limit status")
print("   9. API documentation")

print("\n✅ Exercise 4.1 Complete!")
print("="*80)


EXERCISE 4.1: Testing All Day 59 Components

⏱️  Running comprehensive tests...

🧪 Test Suite - Day 59 Components:

   Test Suite 1: User Authentication

      Test 1.1: User Registration
         ✅ Registration successful: Account created successfully!

      Test 1.2: Duplicate Registration
         ✅ Duplicate correctly rejected: Username already exists

      Test 1.3: User Login
         ✅ Login successful for: test_day59_user

      Test 1.4: Wrong Password
         ✅ Wrong password correctly rejected

      ✅ Authentication Suite Complete

   Test Suite 2: Usage History

      Test 2.1: Add History Entry
         ✅ Entry created: cf1047a8-204f-410c-9bfd-21502146ee89

      Test 2.2: Retrieve History
         ✅ History retrieved: 1 entries

      Test 2.3: Filter by Tool
         ✅ Filtered history: 1 sentiment entries

      Test 2.4: Calculate Analytics
         ✅ Analytics calculated: 1 total queries
            Success rate: 100.0%

      Test 2.5: Delete Entry
         ✅ En

In [22]:
# ==================================================
# EXERCISE 4.2: WHAT I LEARNED TODAY
# ==================================================

print("\n" + "="*80)
print("EXERCISE 4.2: Day 59 Summary")
print("="*80)

print("""
📚 WHAT I LEARNED TODAY:

✅ User Authentication System:
   • Implemented secure authentication with bcrypt password hashing
   • Created UserManager class with registration, login, password verification
   • Built authentication UI with guest mode, login, and signup forms
   • Session state management for logged-in users
   • Password security: bcrypt with automatic salt generation (12 rounds)
   • Base64 encoding for JSON storage compatibility
   • Validation: username length (3+), password strength (6+)

✅ Usage History Tracking:
   • Designed comprehensive history system with per-user storage
   • HistoryManager class: add, retrieve, filter, delete entries
   • History storage: JSON files per user (username.json)
   • Entry structure: id, timestamp, tool, input, result, processing_time, success
   • Query operations: filter by tool, date range, search
   • Recent history sidebar (5 most recent)
   • Full history page with filters and export
   • Re-run and delete functionality

✅ Analytics Dashboard:
   • Summary metrics: total queries, success rate, avg time, favorite tool
   • Visualizations: Bar chart (queries by tool), Pie chart (success/fail)
   • Activity timeline: Line chart showing queries over time
   • Performance table: Statistics per tool
   • Data aggregation: Counter for tools, datetime grouping for timeline
   • Plotly charts with green theme (#4CAF50)

✅ API Key Management:
   • APIKeyManager class for key lifecycle management
   • Key format: 'sk_' + 40 random characters (32 bytes → base64)
   • Cryptographically secure generation using secrets module
   • Key storage: Separate api_keys.json for fast lookup
   • Two-way mapping: user → key, key → user
   • Usage tracking: calls_count, last_used, tools_used breakdown
   • Key regeneration: deactivate old, create new
   • Show/hide toggle in UI for security

✅ Rate Limiting System:
   • RateLimiter class with fixed-window approach
   • Tiered limits: Guest (10/hour), Registered (100/hour), Pro (1000/hour)
   • Window-based enforcement: Reset every hour
   • Usage tracking: count, window_start, limit per user
   • Status display: Progress bar, metrics (used/remaining/limit)
   • Warning at 80% usage threshold
   • Reset time countdown display

✅ API Documentation:
   • Comprehensive developer documentation
   • Sections: Quick start, authentication, endpoints, rate limits, errors
   • Code examples: Python, JavaScript, cURL
   • Interactive expandable endpoint docs
   • Request/response format samples
   • Error code reference table
   • Copy-paste ready examples

📊 PROJECT STATISTICS:

Week 9 Progress:
   • Day 59 complete: 43% (3/7 days)
   • All platform features implemented
   • Tomorrow: Performance optimization

System Features:
   • User authentication: ✅ Complete
   • Usage history: ✅ Complete
   • Analytics: ✅ Complete
   • API keys: ✅ Complete
   • Rate limiting: ✅ Complete
   • API docs: ✅ Complete

Technical Implementation:
   • Classes created: 4 (UserManager, HistoryManager, APIKeyManager, RateLimiter)
   • Storage files: 3 types (users.json, history/*.json, api_keys.json)
   • Security: bcrypt hashing, secrets module for keys
   • UI functions: 8+ (auth UI, history display, analytics, API docs)
   • Data persistence: Local JSON files

Files Structure:
   • user_data/
      - users.json (user accounts)
      - history/{username}.json (per-user history)
      - api_keys/api_keys.json (key → user mapping)
      - rate_limits/rate_limits.json (usage tracking)

💡 KEY INSIGHTS:

1. Authentication requires multiple security layers
   → Bcrypt for passwords (salt + hash + slow)
   → Session state for frontend
   → Base64 encoding for JSON storage
   → Never store plain passwords

2. History tracking enables powerful user features
   → Re-run past queries (productivity)
   → Analytics show usage patterns
   → Export enables data portability
   → Filters make large histories manageable

3. API keys separate authentication concerns
   → Passwords for humans (UI login)
   → Keys for programs (API access)
   → Different lifecycle (rotate vs change)
   → Track usage independently

4. Rate limiting protects system resources
   → Prevents abuse and overuse
   → Fair allocation across users
   → Simple fixed-window sufficient for demo
   → Production: Consider token bucket or sliding window

5. Good documentation is as important as code
   → Developers can't use what they don't understand
   → Examples > explanations
   → Multiple languages serve more users
   → Quick start lowers barrier to entry

6. Local JSON storage works for portfolios
   → Simple to implement and understand
   → No database setup required
   → Easy to inspect and debug
   → Production: Upgrade to PostgreSQL/MongoDB

7. Guest mode removes barriers to adoption
   → Try before signup
   → Zero friction for demos
   → Can upgrade to registered later
   → Balances access with features

8. UI polish matters for user management
   → Show/hide sensitive data (API keys)
   → Progress bars for visual feedback
   → Clear metrics and stats
   → Professional appearance builds trust
""")

print("="*80)


EXERCISE 4.2: Day 59 Summary

📚 WHAT I LEARNED TODAY:

✅ User Authentication System:
   • Implemented secure authentication with bcrypt password hashing
   • Created UserManager class with registration, login, password verification
   • Built authentication UI with guest mode, login, and signup forms
   • Session state management for logged-in users
   • Password security: bcrypt with automatic salt generation (12 rounds)
   • Base64 encoding for JSON storage compatibility
   • Validation: username length (3+), password strength (6+)

✅ Usage History Tracking:
   • Designed comprehensive history system with per-user storage
   • HistoryManager class: add, retrieve, filter, delete entries
   • History storage: JSON files per user (username.json)
   • Entry structure: id, timestamp, tool, input, result, processing_time, success
   • Query operations: filter by tool, date range, search
   • Recent history sidebar (5 most recent)
   • Full history page with filters and export
   • Re-run 

In [23]:
# ==================================================
# EXERCISE 4.3: TOMORROW'S PLAN
# ==================================================

print("\n" + "="*80)
print("EXERCISE 4.3: Tomorrow's Plan")
print("="*80)

print("""
🎯 DAY 60: PERFORMANCE OPTIMIZATION (December 27, 2024)

What I'll do:

1. Profile Application Performance (1 hour)
   • Identify bottlenecks with cProfile
   • Measure inference times per tool
   • Check memory usage patterns
   • Find slow file operations
   • Analyze batch processing performance
   • Test with large datasets (1000+ items)

2. Optimize Model Loading & Inference (1.5 hours)
   • Implement model caching strategies
   • Optimize tokenizer initialization
   • Batch encoding where possible
   • Reduce model forward passes
   • Consider quantization for smaller models
   • Test GPU vs CPU performance
   • Target: <2s inference time per request

3. Optimize Data Operations (1 hour)
   • Improve JSON file read/write
   • Implement lazy loading for history
   • Cache frequently accessed data
   • Optimize pandas operations
   • Reduce unnecessary reruns
   • Pagination for large history

4. Frontend Performance (0.5 hours)
   • Minimize widget rerenders
   • Use st.cache_data where applicable
   • Optimize visualizations
   • Reduce progress bar updates
   • Test responsiveness
   • Memory leak checks

Expected outcomes:
   • Inference time: <2 seconds per request
   • Batch processing: >50 items/second
   • UI responsiveness: No lag
   • Memory stable: No leaks
   • File operations: <100ms
   • Overall app load: <5 seconds

Tech Stack:
   • cProfile (profiling)
   • memory_profiler (memory analysis)
   • time module (benchmarking)
   • Streamlit caching (@st.cache_data, @st.cache_resource)

Time estimate: 4 hours

Success Criteria:
   ✅ All tools respond in <2 seconds
   ✅ Batch processing optimized (>50 items/sec)
   ✅ No UI lag or freezing
   ✅ Memory usage stable
   ✅ File I/O optimized
   ✅ App loads quickly (<5s)
   ✅ Ready for Day 61 (Analytics Dashboard)
""")

print("="*80)


EXERCISE 4.3: Tomorrow's Plan

🎯 DAY 60: PERFORMANCE OPTIMIZATION (December 27, 2024)

What I'll do:

1. Profile Application Performance (1 hour)
   • Identify bottlenecks with cProfile
   • Measure inference times per tool
   • Check memory usage patterns
   • Find slow file operations
   • Analyze batch processing performance
   • Test with large datasets (1000+ items)

2. Optimize Model Loading & Inference (1.5 hours)
   • Implement model caching strategies
   • Optimize tokenizer initialization
   • Batch encoding where possible
   • Reduce model forward passes
   • Consider quantization for smaller models
   • Test GPU vs CPU performance
   • Target: <2s inference time per request

3. Optimize Data Operations (1 hour)
   • Improve JSON file read/write
   • Implement lazy loading for history
   • Cache frequently accessed data
   • Optimize pandas operations
   • Reduce unnecessary reruns
   • Pagination for large history

4. Frontend Performance (0.5 hours)
   • Minimize widget r

In [25]:
print("\n" + "="*80)
print("DAY 59 COMPLETE! ✅")
print("="*80)

print("""
OBJECTIVES ACHIEVED:

✅ Implemented complete user authentication system
   • UserManager class with secure bcrypt hashing
   • Registration with validation (username 3+, password 6+)
   • Login with password verification
   • Guest mode for barrier-free access
   • Session state management
   • Authentication UI (login/signup forms)

✅ Built comprehensive usage history tracking
   • HistoryManager class with full CRUD operations
   • Per-user JSON storage (persistent across sessions)
   • History entries: id, timestamp, tool, input, result, stats
   • Filters: by tool, by date, search in input
   • Actions: re-run, delete, clear all, export CSV
   • Recent history sidebar (5 most recent)
   • Full history page with all entries

✅ Created analytics dashboard
   • Summary metrics (total, success rate, avg time, favorite)
   • Bar chart: Queries by tool
   • Pie chart: Success vs failed
   • Line chart: Activity timeline
   • Performance table: Stats per tool
   • Beautiful Plotly visualizations with green theme

✅ Developed API key management system
   • APIKeyManager class for key lifecycle
   • Secure key generation (secrets + base64)
   • Format: 'sk_' + 40 random characters
   • Key validation and usage tracking
   • Regeneration with old key deactivation
   • Show/hide toggle in UI
   • Usage statistics per key

✅ Implemented rate limiting system
   • RateLimiter class with fixed-window approach
   • Tiered limits: Guest (10/hr), Registered (100/hr)
   • Automatic window reset every hour
   • Usage tracking with remaining count
   • Status display with progress bar
   • Warning at 80% usage threshold

✅ Created comprehensive API documentation
   • Quick start guide
   • Authentication examples
   • Endpoint documentation (all tools)
   • Rate limit information
   • Error code reference
   • Code examples (Python, JS, cURL)
   • Professional developer experience

✅ Complete testing suite
   • Authentication tests (registration, login, validation)
   • History tests (add, retrieve, filter, analytics)
   • API key tests (generate, validate, track, regenerate)
   • Rate limit tests (enforcement, stats, tiers)
   • 35+ automated tests
   • 100% pass rate

📊 KEY METRICS:

Development Time: ~4 hours
   • Part 1 (Authentication): 1.5 hours
   • Part 2 (History & Analytics): 1.5 hours
   • Part 3 (API & Rate Limiting): 1 hour
   • Part 4 (Testing & Docs): 0.5 hours (notebook)

Code Statistics:
   • Classes created: 4 major classes
   • Functions: 25+ (managers + UI)
   • Lines of code: ~1500+ (notebook only)
   • Storage files: 4 types
   • Test cases: 35+

System Features:
   • User accounts: ✅ Fully functional
   • History tracking: ✅ Complete
   • Analytics: ✅ Interactive dashboard
   • API keys: ✅ Generated and managed
   • Rate limiting: ✅ Enforced
   • Documentation: ✅ Professional quality

Data Persistence:
   • users.json: User accounts
   • history/*.json: Per-user history
   • api_keys.json: API key mappings
   • rate_limits.json: Usage tracking
   • All stored locally (user_data/)

💡 KEY LEARNINGS:

1. bcrypt hashing is essential for password security
2. Two-way mappings enable fast lookups (user↔key)
3. Fixed-window rate limiting is simple and effective
4. Guest mode removes barriers while preserving features
5. Analytics turn raw data into user insights
6. Good API docs are as important as the API itself
7. Local JSON storage works great for demos/portfolios
8. Session state management is key to Streamlit apps

🎯 TOMORROW (DAY 60):

Main Goals:
   • Profile application performance 📊
   • Optimize model inference (<2s) ⚡
   • Improve data operations 💾
   • Enhance frontend responsiveness 🎨
   • Eliminate bottlenecks 🔧

Expected Completion: Production-ready performance, <2s inference

💾 FILES CREATED TODAY:

1. day59_user_management_api_features.ipynb
   • Complete Day 59 documentation (22 cells)
   • User authentication system
   • Usage history & analytics
   • API key management
   • Rate limiting system
   • API documentation
   • Comprehensive testing
   • Location: week_9_streamlit_nlp_platform/

2. Data directories created:
   • user_data/ (base directory)
   • user_data/history/ (history files)
   • user_data/api_keys/ (API key mappings)
   • user_data/rate_limits/ (rate limit tracking)

3. (Actual app integration to be done in textai_studio_app.py)
   • Integrate UserManager
   • Add HistoryManager
   • Connect APIKeyManager
   • Implement RateLimiter
   • Add authentication UI
   • Display history & analytics
   • Location: week_8_transformers_advanced_nlp/streamlit_app/

📈 WEEK 9 PROGRESS: 43% (3/7 days)

🎊 Platform features complete! User system ready! 🎊

Day 59 Achievement Unlocked:
✅ User authentication with security
✅ Complete history tracking system
✅ Analytics dashboard with charts
✅ API key management
✅ Rate limiting protection
✅ Professional API documentation
✅ Comprehensive testing
✅ Production-ready user features
""")

print("="*80)


DAY 59 COMPLETE! ✅

OBJECTIVES ACHIEVED:

✅ Implemented complete user authentication system
   • UserManager class with secure bcrypt hashing
   • Registration with validation (username 3+, password 6+)
   • Login with password verification
   • Guest mode for barrier-free access
   • Session state management
   • Authentication UI (login/signup forms)

✅ Built comprehensive usage history tracking
   • HistoryManager class with full CRUD operations
   • Per-user JSON storage (persistent across sessions)
   • History entries: id, timestamp, tool, input, result, stats
   • Filters: by tool, by date, search in input
   • Actions: re-run, delete, clear all, export CSV
   • Recent history sidebar (5 most recent)
   • Full history page with all entries

✅ Created analytics dashboard
   • Summary metrics (total, success rate, avg time, favorite)
   • Bar chart: Queries by tool
   • Pie chart: Success vs failed
   • Line chart: Activity timeline
   • Performance table: Stats per tool
   • Bea